# 12 · An avatar from one ordinary video

From a video to an `.avatar` file in one notebook, to see what footage that is no turntable
gives: one person moving freely, with hair, the face a few dozen pixels, no head videos.

| Part | What | Reuses |
| --- | --- | --- |
| 1 Triage | every moment of the video scored, cheaply; the shots good enough | notebook 11 (`preprocess.triage`) |
| 2 Preparation | the big models on every piece of every good shot: outline, keypoints, hands, face, body parts | notebook 10 (`preprocess.prepare`) |
| 3 Body | Anny's shape and a full pose per frame | the motion stage's fit, alternating with a shape fit |
| 4 Head | ICT-FaceKit's identity and a face per frame, from the face landmarks | the fitting stage's head fit |
| 5 Avatar | the head placed on the body and joined at the neck | the fitting stage |
| 6 Refinement | rendered and compared with every pixel: albedo, light, poses, face, body and head shape | notebooks 08 and 09 |
| 7 Export | `outputs/export/<name>.avatar` for the viewer | notebook 09 |

**Shots**: every stretch of a shot whose triage samples score at least `MIN_SAMPLE_SCORE` is
prepared (a long shot is rarely good all through), cut into pieces of `PIECE_FRAMES` frames. In each piece the largest person is followed, the triage's main
person. The pieces share one body shape and one head; motion is smooth within a piece, not
across.

**Hair** is no part of the avatar (its scalp is the mesh), so the pixels Sapiens2 labels as
hair (`IGNORED_PARTS`) are left out of everything: the silhouettes of every fit, the colour
comparison and the texture bake. Where hair covers the scalp, the scalp's texture is filled in
from its surroundings and its shape comes from the head model's prior.

**The head** is refined together with the body, notebook 08's way folded into 09's: the
identity and smooth shape changes of the head, a face per frame, matched to the face landmarks
as well as the pixels. What 08 also learns needs head videos: her own blendshapes and wrinkle
maps are left out, the eyes are the viewer's own (`STOCK_EYES_AND_MOUTH`).

Everything goes to `outputs/video_avatar/<name>/`; a step named in `REUSE` starts from what an
earlier run stored there.

In [ ]:
import dataclasses
import logging
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import nvdiffrast.torch as dr
import torch
import torch.nn.functional as F

from avatar_splats.body_models import ict
from avatar_splats.body_models.anny import AnnyBody
from avatar_splats.body_models.hybrid import (
    ATLAS_TILES,
    BODY_TEXTURE,
    HANDS,
    HEAD_TEXTURE,
    ICT_CHIN,
    Avatar,
    HeadPlacement,
    build_avatar,
    match_heads,
    rest_body,
    rest_head,
)
from avatar_splats.body_models.ict import IctFaceKit
from avatar_splats.body_models.ict import landmark_vertices as ict_landmark_vertices
from avatar_splats.body_models.measurements import (
    BodyMeasurements,
    fit_measurements,
    measurement_loss,
)
from avatar_splats.body_models.skinning import (
    linear_blend_skinning,
    skinned_point_weights,
    skinned_points,
    sparsify_weights,
)
from avatar_splats.cameras import Camera
from avatar_splats.config import load_config
from avatar_splats.data.video import read_frames
from avatar_splats.export import (
    avatar_arrays,
    eye_metadata,
    face_hair_metadata,
    grow_face_hair,
    measure_eyes,
    measure_hair_colours,
    write_avatar,
)
from avatar_splats.fitting import BodyFit, HeadFit, fit_heads
from avatar_splats.fitting.body import (
    BONE_INFLUENCES,
    MIN_HAND_CONFIDENCE,
    MIN_KEYPOINT_CONFIDENCE,
    POSED_BONES,
    HandJoints,
    badly_fitted,
    surface_keypoint_vertices,
)
from avatar_splats.fitting.head import landmark_targets
from avatar_splats.fitting.losses import (
    depth_loss,
    keypoint_depth_loss,
    keypoint_loss,
    normal_loss,
    second_difference,
    silhouette_loss,
)
from avatar_splats.fitting.placement import place_head
from avatar_splats.fitting.silhouette import (
    SurfaceSamples,
    overlay,
    render_silhouette,
)
from avatar_splats.gaussians import GaussianRecord
from avatar_splats.motion.fit import FACE_SHARE, fit_motion, lifted_frames
from avatar_splats.motion.observe import Observations, mask_size
from avatar_splats.motion.placement import MIN_KEYPOINTS
from avatar_splats.motion.priors import BodyProxies, JointLimits, self_overlap
from avatar_splats.preprocess import Preprocessed
from avatar_splats.preprocess.face import download_face_landmarker
from avatar_splats.preprocess.hands import download_hand_landmarker
from avatar_splats.preprocess.keypoints import (
    COCO_KEYPOINTS,
    EXTRA_KEYPOINTS,
    coco_keypoints,
    estimate_sapiens_keypoints,
    extra_keypoints,
    hand_landmarks,
)
from avatar_splats.preprocess.prepare import (
    NOT_THEM,
    PART_NAMES,
    BodyParts,
    PreparedShot,
    Preparer,
    PrepareSettings,
    boxed,
    is_prepared,
    skin_coloured,
    zoom_cuts,
)
from avatar_splats.preprocess.surface import (
    SurfaceMaps,
    SurfaceSettings,
    estimate_surfaces,
    lift_keypoints,
)
from avatar_splats.preprocess.triage import (
    CLOTHING,
    FACTORS,
    Triage,
    TriageSettings,
    decode_samples,
    good_stretches,
    rank_shots,
    shot_pieces,
    triage,
)
from avatar_splats.rendering import rasterize
from avatar_splats.rendering.surface import render_surface, surface_values
from avatar_splats.training.textures import TextureRecord

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
CONFIG = "../configs/default.yaml"
config = load_config(CONFIG)
device = "cuda"

## Settings

The fitting stage's and the motion stage's settings come from the config; the camera's field
of view replaces both's (one camera cannot tell size from distance either, so `BODY_HEIGHT`
sets the scale). Lower `RESOLUTION` or `CHUNK` if the GPU runs out of memory in the
refinement, `PIECE_FRAMES` in the preparation.

In [ ]:
VIDEO = Path("../examples/live-performance/water.mp4")  # relative to notebooks/
NAME = VIDEO.stem
# Steps that start from an earlier run's results, where stored: "triage", "prepare",
# "keypoints", "body", "surface", "head", "avatar". The triage's samples are decoded again
# either way (it is fast).
REUSE = {}

# Part 1: triage (notebook 11's settings).
TRIAGE = TriageSettings(
    sampling="keyframes",  # or "every_n" (decodes every frame)
    every_n=25,
    max_keyframe_gap=6.0,  # seconds: sparser keyframes fall back to "every_n"
    max_samples=5000,
    analysis_side=640,
    pose_model="../assets/yolo/yolo11m-pose.pt",  # downloads on first use; -n/-s/-l/-x too
    parts_model="facebook/sapiens2-seg-1b",
    parts_size=(512, 384),  # half Sapiens2's native input: about 4x faster
    batch=16,
    min_person=0.5,
    min_height=0.25,  # the main person's box height, in frame heights, to be worth anything
    good_height=0.6,  # and to count fully
    min_joints=6,
    joint_seen=0.5,
    edge=0.01,
    min_sharpness=0.5,  # samples less sharp than this share of the video's median: left out
    context=1.6,
    max_contact=0.3,
    shot_cut=0.45,
)
GOOD, OK = 0.6, 0.3  # sample scores labelled good and ok in the plots
MIN_SAMPLE_SCORE = 0.9  # the stretches of a shot whose samples all score this much are prepared
MAX_DIP = 1  # worse samples in a row a stretch bridges (a passing arm), 0: none
MIN_SHOT_SECONDS = 1.0  # shorter stretches are not prepared
MAX_SHOTS = None  # the best this many of them; None: all

# Part 2: preparation (notebook 10's settings).
FRAME_STEP = 5  # every this many frames of a shot
PIECE_FRAMES = 120  # frames per piece: a piece's frames and labels are held in memory
MIN_PIECE_FRAMES = 12  # a shot's shorter tail is dropped
MAX_SIDE = 1280  # pixels: longer sides are scaled down to this
PREPARE = PrepareSettings(
    min_detection=0.5,
    min_iou=0.3,  # a box overlapping the last one this much is the same person
    isolate_margin=0.02,  # her box widened by this share before everything outside is greyed
    pose_model=config.preprocess.pose_estimator,  # ViTPose; e.g. usyd-community/vitpose-plus-large
    keypoint_seen=0.3,  # a keypoint this sure counts as placed
    min_placed=3,  # keypoints a frame needs placed: 3 keeps close-ups of the head (nose, eyes)
    min_face_placed=5,  # of them on the face: all of nose, eyes and ears; where these are all
    # that is placed, MediaPipe has to find the whole face too
    pose_batch=16,
    min_keypoint=0.3,
    landmark_crop=1.6,
)

# Part 3 on: the fit.
MAX_FIT_FRAMES = 150  # the best usable frames fitted: most skin showing, sharpest
POOL_FRAMES = 300  # the best by ViTPose get Sapiens2's body parts; the best of them by skin fit
MAX_OVERLAP = 0.5  # frames whose box another person's covers this much count for nothing
BODY_PARTS_MODEL = "facebook/sapiens2-seg-1b"  # her outline, hair and clothes; also -5b
BODY_PARTS_BATCH = 8
PERSON_MARGIN = 0.08  # the body parts' crop around her box, in shares of its larger side
MIN_SEGMENT_FRAMES = 5  # pieces with fewer usable frames are left out of the fit
MIN_FACE_FRAMES = 3  # pieces with fewer faces found give the head fit nothing
FACE_OUTLIER = 2.5  # faces the head, placed on their landmarks, misses by more than this many
# times the median are left out (MediaPipe's mesh on faces seen from below); None: all kept
FOV_Y_DEGREES = None  # degrees; None: the median of the estimates (Depth Pro, Sapiens2's
# pointmap), each their median over the frames
BODY_HEIGHT = config.fitting.body_height  # metres
# Sapiens2's labels left out of every comparison and the texture: hair (the scalp is the mesh)
# and clothes, whose texels are filled from frames that show the skin, or from around them.
IGNORED_PARTS = (
    "Hair", "Apparel", "Upper_Clothing", "Lower_Clothing", "Left_Shoe", "Right_Shoe",
    "Left_Sock", "Right_Sock", "Eyeglass",
)  # fmt: skip
SKIN_DISTANCE = 3.0  # clothing this close to the frame's skin colour (in standard deviations)
# is skin Sapiens2 mislabelled, and is fitted
IGNORE_MARGIN = 4  # pixels the ignored parts are widened by: their edges blend into the skin
FACE_PARTS = ("Face_Neck", "Upper_Lip", "Lower_Lip", "Upper_Teeth", "Lower_Teeth", "Tongue")

# The keypoints of the frames to fit, by Sapiens2's pose model over ViTPose's (which chose the
# frames: Sapiens2 is slower, but finds limbs ViTPose gives up on); None: ViTPose's.
POSE_MODEL = "facebook/sapiens2-pose-1b"
# Fingers posed per frame from Sapiens2's hand points (from an open, relaxed hand, within
# their limits, pulled back to it where the points are unsure), or held relaxed.
POSE_FINGERS = True
# Which way the surface faces and how far it is, on the frames to fit: Sapiens2's normals and
# pointmap on the person, Depth Pro's depth of the whole frame (and both's field of view).
SURFACE = SurfaceSettings(
    normal_model="facebook/sapiens2-normal-1b",  # also -5b
    pointmap_model="facebook/sapiens2-pointmap-1b",
    depth_model="apple/DepthPro-hf",
    normal_size=(768, 576),  # the normal model's input: 1024 by 768 is its own
    batch=4,
    depth_batch=2,
    person_margin=0.08,
    map_side=512,  # pixels along the maps' longer side
)

# The body: the motion fit poses a shape, the shape fit reshapes it under those poses, in turns.
MOTION_ITERATIONS = [400, 400, 300]  # the motion fit's steps per phase, at most; it stops a
# phase early where the loss no longer falls
# What is known of the performer (the config's performer section): Anny phenotype values held
# through the fit, and her measurements, which the body is shaped to before any outline is seen
# and held to through the shape fit. The outlines alone made her far too heavy (70 kg, a 90 cm
# waist): weight is the cheapest way to widen every limb at once.
KNOWN_PHENOTYPE = config.performer.phenotype
MEASUREMENTS = {"height": BODY_HEIGHT, **config.performer.measurements}
MEASUREMENT_WEIGHT = 100.0  # per squared relative error: 2 % off weighs as a keypoint term
SHAPE_ROUNDS = 1  # rounds of motion fit and shape fit; a last motion fit follows; each round
# costs a motion fit of every piece
SHAPE_ITERATIONS = 400
SHAPE_LEARNING_RATES = {"shape": 0.01, "pose": 0.002}
# The fitted body's normals and depth against the estimated ones: which way limbs turn and which
# is in front, which keypoints and outlines do not say. In the motion fit (from its second phase
# on), the shape fit and a last pose fit, those two on SURFACE_FRAMES frames per step.
# The confident keypoints lifted onto the pointmap: the motion fit starts from them, and every
# fit keeps them in their order in depth.
SURFACE_WEIGHTS = {"normals": 0.1, "depth": 0.5, "keypoint_depth": 1.0}  # per 1 - cos, metre
SURFACE_FRAMES = 32
# Frames the first motion fit fits much worse than the rest (IoU more than DROP_SPREAD robust
# spreads below the median, at most the worst DROP_MOST) are left out of the shape fit and the
# refinement: her shape and texture come from frames her pose fits. They keep their pose.
DROP_SPREAD = 2.0
DROP_MOST = 1 / 3
CORRECTION_WEIGHT = 1.0  # per radian² of the shape fit's pose corrections, and per decimetre²
POSE_CORRECTION_WEIGHT = 0.1  # the same in the last pose fit: looser, so the surface can turn
# what the motion fit left

# The refinement (notebook 09's, with 08's head folded in).
RESOLUTION = 1024  # pixels along the frames' longer side, at most the prepared frames'
BODY_TEXTURE_SIZE = 1024
HEAD_TILE_SIZE = 512
REBAKE_EVERY = 50
MIN_SHADING = 0.2
FILL_WEIGHT = 0.05
STOCK_EYES_AND_MOUTH = True  # the viewer's own eyes and mouth: a small face bakes poor ones
HOLD_OUT = 8
CHUNK = 9
STEP_FRAMES = 48  # frames drawn at random for each refinement step; all are baked and scored
SMOOTHING_STEPS = 60  # the body's shape changes
HEAD_SMOOTHING_STEPS = 40  # the head's (notebook 08's broad layer)
FROZEN_NEAR = 0.03
EDGE_BAND = 4
NEAR, FAR = 0.1, 20.0  # metres

ITERATIONS = {"appearance": 300, "pose": 400, "shape": 400}

LEARNING_RATES = {
    "light": 0.01,
    "pose": 0.001,
    "expressions": 0.02,
    "identity": 0.01,
    "offsets": 0.02,  # millimetres per step, before smoothing
    "head_offsets": 0.02,
}

WEIGHTS = {
    "photo": 1.0,
    "silhouette": 1.0,
    "keypoints": 1.0,  # the 17 body keypoints, in image heights (keypoint_loss)
    "landmarks": 1.0,  # the head model's 68 landmarks against MediaPipe's
    "pose_prior": 1.0,
    "temporal": 1.0,
    "joint_limits": config.motion.joint_limit_weight,  # as in the body fit
    "sparsity": config.fitting.expression_sparsity_weight,
    "identity_prior": config.fitting.identity_prior_weight,
    "offsets": 1e-3,
    "head_offsets": 1e-3,
    "normals": 0.1,  # per 1 - cosine against the estimated normals
    "depth": 0.5,  # per metre against the estimated depths, scaled to agree in median
}

out_dir = config.outputs / "video_avatar" / NAME
out_dir.mkdir(parents=True, exist_ok=True)
timings = {}


def timed(step, started):
    timings[step] = timings.get(step, 0.0) + time.time() - started

## Part 1: triage

Notebook 11's pass: the keyframes (or every `every_n`-th frame), YOLO-pose on all of them, the
contact check with Sapiens2 on those with someone large and visible enough; each sample's score
multiplies five shares (size, joints, whole, sharpness, alone) and its lowest ones are its
reasons.

In [ ]:
started = time.time()
samples = decode_samples(VIDEO, TRIAGE)
triage_path = out_dir / "triage.npz"
if "triage" in REUSE and triage_path.exists():
    triaged, contact_crops = Triage.load(triage_path), {}
    print("triage read from", triage_path)
else:
    triaged, contact_crops = triage(samples, TRIAGE, device)
    triaged.save(triage_path)
torch.cuda.empty_cache()
timed("triage", started)
SAMPLE_COUNT = len(samples.frames)
SAMPLE_WIDTH, SAMPLE_HEIGHT = (int(v) for v in triaged.sample_size)
grade = np.where(triaged.score >= GOOD, "good", np.where(triaged.score >= OK, "ok", "bad"))
gaps = np.diff(triaged.times)
print(
    f"{VIDEO.name}: {samples.video_size[0]}x{samples.video_size[1]}, {triaged.fps:g} fps, "
    f"{triaged.duration:.0f} s; {SAMPLE_COUNT} samples ({samples.sampling}), "
    f"every {np.median(gaps):.1f} s; {triaged.shot[-1] + 1} shots"
)
for name in ("good", "ok", "bad"):
    print(f"{name:>5}: {(grade == name).sum():5d} samples ({(grade == name).mean():.0%})")

Scores and their shares over time (grey lines: shot cuts), then the best and the worst samples:
the main person's box (green) and joints, other people's body pixels where the contact check ran
(magenta).

In [ ]:
minutes = triaged.times / 60
fig, axes = plt.subplots(len(FACTORS) + 1, 1, figsize=(16, 2 * (len(FACTORS) + 1)), sharex=True)
for name, colour in {"good": "tab:green", "ok": "tab:orange", "bad": "tab:red"}.items():
    picked = grade == name
    axes[0].scatter(minutes[picked], triaged.score[picked], s=6, color=colour, label=name)
for cut_at in minutes[1:][np.diff(triaged.shot) > 0]:
    axes[0].axvline(cut_at, color="grey", lw=0.3)
axes[0].set_ylabel("score")
axes[0].legend(loc="upper right")
for ax, k in zip(axes[1:], range(len(FACTORS)), strict=True):
    ax.plot(minutes, triaged.factors[:, k], lw=0.6)
    ax.set_ylabel(FACTORS[k])
    ax.set_ylim(-0.05, 1.05)
axes[-1].set_xlabel("minutes")
plt.tight_layout()
plt.show()

COCO_BONES = ((0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11),
              (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16))  # fmt: skip


def reasons(i):
    if triaged.people[i] == 0:
        return "nobody"
    if not np.isfinite(triaged.contact[i]):
        sharp = triaged.sharpness[i] >= TRIAGE.min_sharpness * np.nanmedian(triaged.sharpness)
        return "too small, or too few joints" if sharp else "blurry"
    weakest = np.argsort(triaged.factors[i])[:2]
    factors = triaged.factors[i]
    return ", ".join(f"{FACTORS[k]} {factors[k]:.2f}" for k in weakest if factors[k] < 0.9) or "-"


def annotated(i):
    image = samples.frames[i].copy()
    if i in contact_crops:
        (x0, y0, x1, y1), parts = contact_crops[i]
        others = ((parts > 0) & ~np.isin(parts, CLOTHING)).astype(np.uint8)
        others = cv2.resize(others, (x1 - x0, y1 - y0), interpolation=cv2.INTER_NEAREST)
        full = np.zeros(image.shape[:2], bool)
        fx0, fy0 = max(x0, 0), max(y0, 0)
        fx1, fy1 = min(x1, SAMPLE_WIDTH), min(y1, SAMPLE_HEIGHT)
        full[fy0:fy1, fx0:fx1] = others[fy0 - y0 : fy1 - y0, fx0 - x0 : fx1 - x0].astype(bool)
        bx0, by0, bx1, by1 = triaged.boxes[i, :4].astype(int)
        full[max(by0, 0) : by1, max(bx0, 0) : bx1] = False
        image[full] = (0.5 * image[full] + 0.5 * np.array([255, 0, 255])).astype(np.uint8)
    if triaged.people[i]:
        x0, y0, x1, y1 = triaged.boxes[i, :4].astype(int)
        cv2.rectangle(image, (x0, y0), (x1, y1), (0, 220, 0), 2)
        points = triaged.keypoints[i]
        for a, b in COCO_BONES:
            c = min(points[a, 2], points[b, 2])
            if c >= 0.3:
                colour = (int(255 * (1 - c)), int(255 * c), 0)
                cv2.line(image, points[a, :2].astype(int), points[b, :2].astype(int), colour, 2)
    return image


def sheet(indices, title, columns=4):
    rows = max(1, -(-len(indices) // columns))
    height = 18 / columns * SAMPLE_HEIGHT / SAMPLE_WIDTH * rows + 0.6 * rows
    fig, axes = plt.subplots(rows, columns, figsize=(18, height), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i in zip(axes.flat, indices, strict=False):
        ax.imshow(annotated(i))
        ax.set_title(f"{minutes[i]:.1f} min · {triaged.score[i]:.2f} · {reasons(i)}", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


ranked = np.argsort(-triaged.score)
sheet(ranked[:12], "best")
sheet([i for i in ranked[::-1] if triaged.people[i] > 0][:12], "worst with someone in them")

## The shots to prepare

Each shot is split into its good stretches (`preprocess.triage.good_stretches`): runs of samples
scoring at least `MIN_SAMPLE_SCORE`, bridging at most `MAX_DIP` worse ones in a row, never across a
cut. A stretch's score is the median of its samples'. Every stretch lasting `MIN_SHOT_SECONDS` is
prepared, best first, cut into pieces of `PIECE_FRAMES` frames; in each the person is picked on its
best triage sample.

In [ ]:
shots = rank_shots(triaged.times, triaged.shot, triaged.score, triaged.duration)
stretches = good_stretches(
    triaged.times, triaged.shot, triaged.score, triaged.duration, MIN_SAMPLE_SCORE, MAX_DIP
)
chosen_shots = [s for s in stretches if s.end - s.begin >= MIN_SHOT_SECONDS][:MAX_SHOTS]
pieces = shot_pieces(
    chosen_shots,
    triaged.times,
    triaged.score,
    triaged.fps,
    FRAME_STEP,
    PIECE_FRAMES,
    MIN_PIECE_FRAMES,
)
if not pieces:
    raise LookupError("no shot is good enough: lower MIN_SAMPLE_SCORE or MIN_SHOT_SECONDS")
print(
    f"{len(chosen_shots)} good stretches in {len(shots)} shots "
    f"(longest shot {max(s.end - s.begin for s in shots):.0f} s), {len(pieces)} pieces:"
)
print(f"{'rank':>4} {'score':>6} {'from':>8} {'to':>8}  pieces")
for rank, shot in enumerate(chosen_shots):
    count = sum(piece.shot == rank for piece in pieces)
    print(f"{rank:4d} {shot.score:6.2f} {shot.begin:7.1f}s {shot.end:7.1f}s  {count}")

## Part 2: preparation

The cheap models on every frame of every piece (`preprocess.prepare`): the detector follows
the largest person from the picked frame (each frame's box overlapping the last one most), and
everything outside that box (widened by `isolate_margin`) is greyed out. Whoever else is inside
it stays: interlocked people would lose parts of her to any cut through it, so each frame stores
how much other people's boxes cover hers, and the frames to fit are chosen away from them.
ViTPose's 17 body keypoints run on what is left; a frame goes on where it places at least
`min_placed` keypoints (`keypoint_seen`), `min_face_placed` of them on the face, so close-ups of
the head and upper body go on too but ones without the face don't, and there MediaPipe's hands
and face landmarks run too. Sapiens2's body parts come later, on the frames to fit only. Each
piece is written to `prepared/piece-<k>.npz`.

In [ ]:
SHOWN = 4  # accepted and rejected frames drawn per piece
ZOOM_MARGIN = 0.15  # the drawn frames are cropped to the person's box, widened by this share
HAND_BONES = ((0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10),
              (10, 11), (11, 12), (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18),
              (18, 19), (19, 20), (0, 17))  # fmt: skip


def confidence_colour(value):
    return (int(255 * (1 - value)), int(255 * value), 0)


def drawn_frame(result, frame, i):
    # the frame greyed outside the person's box (green: accepted, red: rejected), ViTPose's
    # keypoints by confidence, hands (yellow) and face landmarks (cyan)
    image = boxed(frame, result.boxes[i], PREPARE.isolate_margin)
    box = result.boxes[i]
    size = max(box[2] - box[0], box[3] - box[1]) if np.isfinite(box).all() else image.shape[0]
    thickness = max(1, round(size / 250))
    if np.isfinite(box).all():
        colour = (0, 230, 0) if result.usable[i] else (255, 50, 50)
        x0, y0, x1, y1 = box.astype(int)
        cv2.rectangle(image, (x0, y0), (x1, y1), colour, thickness)
    points = result.keypoints[i]
    for a, b in COCO_BONES:
        if np.isfinite(points[[a, b]]).all():
            cv2.line(
                image,
                points[a, :2].astype(int),
                points[b, :2].astype(int),
                confidence_colour(min(points[a, 2], points[b, 2])),
                thickness,
            )
    for x, y, c in points[np.isfinite(points).all(1)]:
        cv2.circle(image, (int(x), int(y)), 2 * thickness, confidence_colour(c), -1)
    for hand in result.hands[i]:
        if np.isfinite(hand).all():
            for a, b in HAND_BONES:
                cv2.line(image, hand[a, :2].astype(int), hand[b, :2].astype(int), (255, 220, 0), 1)
    face_points = result.face_landmarks[i]
    for x, y, _ in face_points[np.isfinite(face_points).all(1)]:
        cv2.circle(image, (int(x), int(y)), 1, (0, 255, 255), -1)
    return zoomed(image, box)


def zoomed(image, box):
    # the image cropped to the box widened by ZOOM_MARGIN; whole where there is no box
    if not np.isfinite(box).all():
        return image
    x0, y0, x1, y1 = box
    pad = ZOOM_MARGIN * max(x1 - x0, y1 - y0)
    height_, width_ = image.shape[:2]
    x0, y0 = max(int(x0 - pad), 0), max(int(y0 - pad), 0)
    x1, y1 = min(int(x1 + pad), width_), min(int(y1 + pad), height_)
    return image[y0:y1, x0:x1]


def why(result, i):
    # the detector's score, how much of the box others cover, the keypoints not placed
    if result.detection[i] == 0:
        return "not found"
    text = f"detected {result.detection[i]:.2f}, others cover {result.overlap[i]:.0%}"
    missing = np.nan_to_num(result.keypoints[i, :, 2]) < PREPARE.keypoint_seen
    if missing.any():
        names = ", ".join(COCO_KEYPOINTS[k] for k in np.flatnonzero(missing))
        text += f"\n{missing.sum()} not placed: {names}"
    return text


def show_piece(result, piece_frames, title):
    found = result.detection > 0
    rows = (
        ("accepted", np.flatnonzero(result.usable)),
        ("rejected", np.flatnonzero(found & ~result.usable)),
    )
    fig, axes = plt.subplots(2, SHOWN, figsize=(18, 12), squeeze=False)
    for row, (label, frames_) in zip(axes, rows, strict=True):
        for ax in row:
            ax.axis("off")
        if len(frames_) > SHOWN:
            frames_ = frames_[np.linspace(0, len(frames_) - 1, SHOWN).round().astype(int)]
        for ax, i in zip(row, frames_, strict=False):
            ax.imshow(drawn_frame(result, piece_frames[i], i))
            ax.set_title(f"{label} {result.frame_indices[i]}: {why(result, i)}", fontsize=8)
        if not len(frames_):
            row[0].set_title(f"no {label} frames", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


prepared_dir = out_dir / "prepared"
prepared_dir.mkdir(exist_ok=True)
models = config.preprocess
prepare_timings = {}
paths_of, preparer = {}, None


def decoded_piece(piece):
    # (frames scaled as preparation scales them, the scale, seconds): run in a background thread,
    # so the next piece decodes while the GPU works on this one
    started_ = time.time()
    piece_frames = list(
        read_frames(VIDEO, step=FRAME_STEP, max_frames=piece.count, start=piece.first)
    )
    piece_scale = min(1.0, MAX_SIDE / max(piece_frames[0].shape[:2]))
    if piece_scale < 1:
        size = tuple(round(v * piece_scale) for v in piece_frames[0].shape[1::-1])
        piece_frames = [cv2.resize(f, size, interpolation=cv2.INTER_AREA) for f in piece_frames]
    return piece_frames, piece_scale, time.time() - started_


def saved(result, path):
    # compressing and writing a piece, in a background thread; returns the seconds it took
    started_ = time.time()
    result.save(path)
    return time.time() - started_


todo = []
for k, piece in enumerate(pieces):
    path = prepared_dir / f"piece-{k:03d}.npz"
    # only where it is this stretch, prepared by this version
    if "prepare" in REUSE and is_prepared(path, piece.first, FRAME_STEP, piece.count):
        paths_of[k] = path
    else:
        todo.append((k, piece, path))
with ThreadPoolExecutor(1) as decoder, ThreadPoolExecutor(1) as saver:
    upcoming = decoder.submit(decoded_piece, todo[0][1]) if todo else None
    saves = []
    for n, (k, piece, path) in enumerate(todo):
        if preparer is None:
            preparer = Preparer(
                PREPARE,
                models.detector,
                download_hand_landmarker(models.hand_landmarker),
                download_face_landmarker(models.face_landmarker),
                device,
            )
        started = time.time()
        piece_frames, piece_scale, seconds = upcoming.result()
        prepare_timings["decode (beside the GPU)"] = (
            prepare_timings.get("decode (beside the GPU)", 0.0) + seconds
        )
        if n + 1 < len(todo):
            upcoming = decoder.submit(decoded_piece, todo[n + 1][1])
        result = preparer.prepare(
            piece_frames,
            piece.first + np.arange(len(piece_frames)) * FRAME_STEP,
            piece_scale,
            min(piece.select, len(piece_frames) - 1),
            piece.shot,
            piece.score,
            prepare_timings,
        )
        timed("preparation", started)
        if result is None:
            print(f"piece {k}: nobody found on its picked frame, left out")
            continue
        saves.append(saver.submit(saved, result, path))
        paths_of[k] = path
        crowded = (np.nan_to_num(result.overlap[result.usable]) >= MAX_OVERLAP).sum()
        print(
            f"piece {k} (shot {piece.shot}, frames {piece.first}..): usable "
            f"{result.usable.sum()} of {len(result.usable)}, a face in "
            f"{np.isfinite(result.face_confidence).sum()}, {crowded} with someone covering "
            f"{MAX_OVERLAP:.0%} of her box ({time.time() - started:.0f} s)"
        )
        show_piece(result, piece_frames, f"piece {k}: shot {piece.shot}, frames {piece.first}..")
        del result, piece_frames
    prepare_timings["save (beside the GPU)"] = sum(future.result() for future in saves)
piece_paths = [paths_of[k] for k in sorted(paths_of)]
# Pieces of an earlier run that are none of these go: later steps and notebook 13 read them all.
stale = [path for path in prepared_dir.glob("piece-*.npz") if path not in piece_paths]
for path in stale:
    path.unlink()
if stale:
    print(f"{len(stale)} pieces of an earlier run removed")
del preparer
torch.cuda.empty_cache()
for step_, seconds in sorted(prepare_timings.items(), key=lambda item: -item[1]):
    print(f"  {step_:<28}{seconds:>8.0f} s")

## The frames to fit

Not every usable frame: first a pool of the best `POOL_FRAMES` by what preparation knows (her
box's size in pixels, so close-ups count for their detail, times the share of body keypoints
ViTPose placed, times how little of her box other people cover, up to `MAX_OVERLAP`, times the
sharpness against the median, at most 1). Sapiens2's body parts run on the pool only
(`BodyParts`): her outline, hair and clothes. Then the best `MAX_FIT_FRAMES` of the pool, ranked
the same way but by the skin they show (body parts that are not `IGNORED_PARTS`) instead of the
box. In both rounds pieces left with fewer than `MIN_SEGMENT_FRAMES` frames, or none the motion
fit can place (`MIN_KEYPOINTS` body keypoints), are dropped and their share goes to the next best
frames. Each piece is a segment, cut again where the face's size jumps (`zoom_cuts`: a cut to a
close-up the shot detection missed, kept for the face): motion is smooth within one, not across.
The ignored pixels are the person's `IGNORED_PARTS` (hair and clothes), widened by
`IGNORE_MARGIN`.

In [ ]:
started = time.time()
# What preparation knows of each usable frame: the person's box in pixels, the share of body
# keypoints placed, how little of the box others cover, the sharpness, whether it can be placed.
qualities = []  # per piece: usable frame indices, box area, keypoints, alone, sharpness, placeable
for path in piece_paths:
    stored = np.load(path)
    usable_ = np.flatnonzero(stored["usable"])
    box = stored["boxes"][usable_]
    confident = stored["keypoints"][usable_, :17, 2] >= MIN_KEYPOINT_CONFIDENCE
    alone = np.clip(1 - np.nan_to_num(stored["overlap"][usable_]) / MAX_OVERLAP, 0, 1)
    qualities.append(
        (
            usable_,
            (box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]),
            confident.mean(-1),
            alone,
            stored["sharpness"][usable_],
            confident.sum(-1) >= MIN_KEYPOINTS,
        )
    )
median_sharpness = np.nanmedian(np.concatenate([q[4] for q in qualities]))


def factor(p_, rows):
    # (n,) keypoints placed x alone x sharpness (at most 1) of piece p_'s usable frames at rows
    _, _, placed, alone, sharp, _ = qualities[p_]
    return placed[rows] * alone[rows] * np.clip(np.nan_to_num(sharp[rows] / median_sharpness), 0, 1)


def choose(candidates, count):
    """{piece: frames} of the best count (quality, piece, frame) candidates; pieces left with
    too few of them (or none the motion fit can place) are dropped and their share goes to the
    next best frames of the others."""
    candidates = sorted(candidates, key=lambda c: -c[0])
    dropped = set()
    for _ in range(5):
        picked = {}
        for _, p_, i in [c for c in candidates if c[1] not in dropped][:count]:
            picked.setdefault(p_, []).append(i)
        too_few = {
            p_
            for p_, frames_ in picked.items()
            if len(frames_) < MIN_SEGMENT_FRAMES
            or not qualities[p_][5][np.isin(qualities[p_][0], frames_)].any()
        }
        if not too_few:
            break
        dropped |= too_few
    return {p_: sorted(frames_) for p_, frames_ in picked.items()}


# Round 1: the pool, by her box in pixels.
pool = choose(
    [
        (q, p_, int(i))
        for p_, (usable_, area, *_) in enumerate(qualities)
        for q, i in zip(area * factor(p_, slice(None)), usable_, strict=True)
    ],
    POOL_FRAMES,
)
columns = {
    key: []
    for key in (
        "frames", "frame_indices", "boxes", "keypoints", "hands", "face_landmarks", "piece",
        "frame",
    )
}  # fmt: skip
for p_, path in enumerate(piece_paths):
    if p_ not in pool:
        continue
    piece = PreparedShot.load(path)
    keep = np.array(pool[p_])  # in time order
    indices = piece.frame_indices
    step = int(indices[1] - indices[0]) if len(indices) > 1 else 1
    decoded = list(read_frames(VIDEO, step=step, max_frames=len(indices), start=int(indices[0])))
    width_, height_ = (int(v) for v in piece.image_size)
    columns["frames"].append(
        np.stack(
            [cv2.resize(decoded[i], (width_, height_), interpolation=cv2.INTER_AREA) for i in keep]
        )
    )
    columns["frame_indices"].append(indices[keep])
    columns["boxes"].append(piece.boxes[keep])
    columns["keypoints"].append(piece.keypoints[keep, :17])  # COCO's 17 come first
    columns["hands"].append(piece.hands[keep])
    columns["face_landmarks"].append(piece.face_landmarks[keep])
    columns["piece"].append(np.full(len(keep), p_))
    columns["frame"].append(keep)
    del piece, decoded
if not columns["piece"]:
    raise LookupError("no piece has enough usable frames")
joined = {key: np.concatenate(values) for key, values in columns.items()}
del columns

# Sapiens2's body parts on the pool.
body_parts_model = BodyParts(BODY_PARTS_MODEL, BODY_PARTS_BATCH, PERSON_MARGIN, device)
pool_parts, _ = body_parts_model(joined["frames"], joined["boxes"])
del body_parts_model
torch.cuda.empty_cache()
timed("body parts", started)

# Round 2: the frames to fit, by the skin they show.
skin_labels = [
    k for k, name in enumerate(PART_NAMES) if name != "Background" and name not in IGNORED_PARTS
]
skin = np.isin(pool_parts, skin_labels).sum(axis=(1, 2)).astype(float)
row_of = {
    (int(p_), int(i)): r
    for r, (p_, i) in enumerate(zip(joined["piece"], joined["frame"], strict=True))
}
fit_frames = choose(
    [
        (skin[r] * factor(p_, [np.searchsorted(qualities[p_][0], i)])[0], p_, i)
        for (p_, i), r in row_of.items()
    ],
    MAX_FIT_FRAMES,
)
rows = np.array([row_of[(p_, i)] for p_ in sorted(fit_frames) for i in fit_frames[p_]])
print(
    f"{len(rows)} frames chosen from a pool of {len(row_of)}, of "
    f"{sum(len(q[0]) for q in qualities)} usable frames, in {len(fit_frames)} of "
    f"{len(piece_paths)} pieces"
)
joined = {key: values[rows] for key, values in joined.items()}
joined["body_parts"] = pool_parts[rows]
del pool_parts

# Segments: one per piece, cut again where a close-up the shot detection missed begins.
segment_paths, segments = [], []
for p_ in sorted(fit_frames):
    in_piece = joined["piece"] == p_
    runs = np.cumsum(zoom_cuts(joined["keypoints"][in_piece], MIN_KEYPOINT_CONFIDENCE))
    segments.append(len(segment_paths) + runs)
    segment_paths += [piece_paths[p_]] * (runs[-1] + 1)
joined["segment"] = np.concatenate(segments)
joined["masks"] = joined["body_parts"] != NOT_THEM
frames, frame_indices, segment = joined["frames"], joined["frame_indices"], joined["segment"]
masks, boxes, keypoints = joined["masks"], joined["boxes"], joined["keypoints"]
hands, face_landmarks, body_parts = joined["hands"], joined["face_landmarks"], joined["body_parts"]
del joined
FRAMES, (HEIGHT, WIDTH) = len(frames), masks.shape[1:]
SEGMENTS = len(segment_paths)
fps = triaged.fps / FRAME_STEP
same_segment = torch.from_numpy(segment[1:] == segment[:-1]).to(device)  # (N - 1,) in one piece
steady = same_segment[1:] & same_segment[:-1]  # (N - 2,) triples in one piece


def widened(flags, radius):
    # (N, H, W) bool, each frame's True pixels grown by radius
    if radius < 1:
        return flags
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    return np.stack([cv2.dilate(f.astype(np.uint8), kernel).astype(bool) for f in flags])


# Clothing whose colour is the skin's is skin Sapiens2 mislabelled (bare legs as
# Lower_Clothing): it is fitted. Hair is always ignored.
labelled = np.isin(body_parts, [PART_NAMES.index(n) for n in IGNORED_PARTS])
clothing = labelled & (body_parts != PART_NAMES.index("Hair"))
freed = np.stack(
    [
        skin_coloured(frames[i], body_parts[i], clothing[i], SKIN_DISTANCE)
        for i in range(len(frames))
    ]
)
print(
    f"clothing: {clothing.sum() / max(masks.sum(), 1):.1%} of the person's pixels, "
    f"{freed.sum() / max(clothing.sum(), 1):.0%} of it coloured like the skin and fitted"
)
ignored = widened(labelled & ~freed, IGNORE_MARGIN)
face = np.isin(body_parts, [PART_NAMES.index(n) for n in FACE_PARTS])
found_faces = np.isfinite(face_landmarks).all(axis=(1, 2))
timed("load", started)
print(f"{FRAMES} frames in {SEGMENTS} pieces, {WIDTH}x{HEIGHT}")
print(f"a face in {found_faces.sum()} frames")
print(f"ignored: {ignored.sum() / max(masks.sum(), 1):.1%} of the person's pixels")

picks = np.linspace(0, FRAMES - 1, 5).round().astype(int)
fig, axes = plt.subplots(1, len(picks), figsize=(18, 18 / len(picks) * HEIGHT / WIDTH))
for ax, i in zip(axes, picks, strict=True):
    picture = frames[i].copy()
    picture[ignored[i]] = (0.4 * picture[ignored[i]] + [150, 0, 150]).astype(np.uint8)
    picture[~masks[i]] //= 3
    ax.imshow(picture)
    ax.set_title(f"piece {segment[i]}, frame {frame_indices[i]}: ignored in magenta", fontsize=9)
    ax.axis("off")
plt.show()

## The surface: normals, pointmap, depth

On the frames to fit (`preprocess.surface`): Sapiens2's surface normals and pointmap on the same
crop around the person as its body parts, and Depth Pro's depth of the whole frame. The
pointmap's points against their pixels give a focal length, and Depth Pro estimates one: two
estimates of the field of view, compared here per piece (a zoom between
shots shows as pieces apart). Below, per frame: the frame, the estimated normals (red: facing
right, green: up, blue: the camera), the pointmap's depth and Depth Pro's on the person (near:
dark).


In [ ]:
started = time.time()
surface_keys = [
    f"{Path(segment_paths[piece]).stem}/{index}"
    for piece, index in zip(segment, frame_indices, strict=True)
]

# Sapiens2's keypoints, over ViTPose's: the 17 COCO ones, the hands (over MediaPipe's) and
# the extra ones on the feet, elbows and shoulders, which ViTPose has none of.
extra = np.full((FRAMES, len(EXTRA_KEYPOINTS), 3), np.nan)
if POSE_MODEL is not None:
    pose_path = out_dir / "keypoints.npz"
    stored_pose = np.load(pose_path) if "keypoints" in REUSE and pose_path.exists() else None
    if (
        stored_pose is not None
        and list(stored_pose["keys"]) == surface_keys
        and stored_pose["keypoints"].shape[1] > len(COCO_KEYPOINTS)
    ):
        goliath = stored_pose["keypoints"]
        print("keypoints read from", pose_path)
    else:
        started = time.time()
        goliath = estimate_sapiens_keypoints(frames, boxes, POSE_MODEL, device)
        np.savez(pose_path, keys=np.array(surface_keys), keypoints=goliath)
        timed("keypoints", started)
    better, extra = coco_keypoints(goliath), extra_keypoints(goliath)
    hands = hand_landmarks(goliath)
    old_sure = keypoints[..., 2] >= MIN_KEYPOINT_CONFIDENCE
    new_sure = better[..., 2] >= MIN_KEYPOINT_CONFIDENCE
    apart = np.linalg.norm(better[..., :2] - keypoints[..., :2], axis=-1) > 0.05 * HEIGHT
    apart &= old_sure & new_sure
    print(
        f"Sapiens2's keypoints: {new_sure.sum(1).mean():.1f} of 17 sure per frame (ViTPose: "
        f"{old_sure.sum(1).mean():.1f}); {apart.sum()} sure in both lie more than 5 % of the "
        f"height apart, in {apart.any(1).sum()} frames"
    )
    keypoints = better
surface_path = out_dir / "surface.npz"
surface = None
if "surface" in REUSE and surface_path.exists():
    surface = SurfaceMaps.load(surface_path)
    if list(surface.keys) != surface_keys:
        print("the stored surface maps are of other frames: estimated again")
        surface = None
    else:
        print("surface maps read from", surface_path)
if surface is None:
    surface = estimate_surfaces(frames, masks, boxes, surface_keys, SURFACE, device)
    surface.save(surface_path)
    timed("surface maps", started)
MAP_WIDTH, MAP_HEIGHT = (int(v) for v in surface.map_size)
print(f"maps at {MAP_WIDTH}x{MAP_HEIGHT}")


def fov_of(focal):
    return np.degrees(2 * np.arctan(HEIGHT / 2 / np.asarray(focal, float)))


focals = {"Depth Pro": surface.depth_focal, "Sapiens2 pointmap": surface.point_focal}
fov_estimates = {}
fig, ax = plt.subplots(figsize=(16, 3))
for name, focal in focals.items():
    per_frame = fov_of(focal)
    fov_estimates[name] = float(np.nanmedian(per_frame))
    per_piece = [np.nanmedian(per_frame[segment == piece]) for piece in range(SEGMENTS)]
    print(
        f"{name:>18}: field of view {fov_estimates[name]:5.1f}° (median), per piece "
        + " ".join(f"{v:.0f}" for v in per_piece)
    )
    ax.plot(per_frame, ".", label=name)
for piece in range(1, SEGMENTS):
    ax.axvline(np.flatnonzero(segment == piece)[0] - 0.5, color="grey", lw=0.5)
ax.set_ylabel("vertical field of view (°)")
ax.set_xlabel("frame to fit (grey lines: pieces)")
ax.legend()
plt.show()
if FOV_Y_DEGREES is None:
    FOV_Y_DEGREES = round(float(np.median(list(fov_estimates.values()))), 1)
print(f"fitted with a field of view of {FOV_Y_DEGREES:g}°")
fitting_config = dataclasses.replace(config.fitting, fov_y_degrees=FOV_Y_DEGREES)
# Steps per phase at most: keypoints, + hands, + silhouettes.
motion_config = dataclasses.replace(
    config.motion,
    fov_y_degrees=FOV_Y_DEGREES,
    iterations=MOTION_ITERATIONS,
    normal_weight=SURFACE_WEIGHTS["normals"],
    depth_weight=SURFACE_WEIGHTS["depth"],
    keypoint_depth_weight=SURFACE_WEIGHTS["keypoint_depth"],
    pose_fingers=POSE_FINGERS,
)

# The estimates on the GPU at the maps' size, half precision; resized where compared.
surface_normals = torch.from_numpy(surface.normals).to(device)
surface_depths = [
    torch.from_numpy(surface.point_depth).to(device),
    torch.from_numpy(surface.depth).to(device),
]
raster_context = dr.RasterizeCudaContext()


def surface_targets(frames_, size):
    """The frames' estimated (n, H, W, 3) normals, zero where unknown, and their (n, H, W)
    depths (the pointmap's, Depth Pro's), NaN where unknown, at size (H, W)."""
    normals = F.interpolate(
        surface_normals[frames_].float().permute(0, 3, 1, 2), size, mode="bilinear"
    ).permute(0, 2, 3, 1)
    known = normals.norm(dim=-1, keepdim=True) > 0.5
    normals = F.normalize(normals, dim=-1) * known
    depths = [
        F.interpolate(d[frames_].float()[:, None], size, mode="bilinear")[:, 0]
        for d in surface_depths
    ]
    return normals, depths


def surface_terms(drawn, normals, depths, region):
    """The normal and depth terms of a SurfaceImage against the estimates, in a region."""
    region = region * drawn.covered * (normals.norm(dim=-1) > 0.5)
    return {
        "normals": normal_loss(drawn.normals, normals, region),
        "depth": sum(depth_loss(drawn.depth, d, region) for d in depths) / len(depths),
    }


def normal_colours(normals):
    # (..., 3) camera-axes normals as RGB: red right, green up, blue towards the camera
    return ((normals * np.array([1, -1, -1]) + 1) / 2).clip(0, 1) * (
        np.abs(normals).sum(-1, keepdims=True) > 0
    )


shown = np.linspace(0, FRAMES - 1, 5).round().astype(int)
fig, axes = plt.subplots(4, len(shown), figsize=(18, 4 * 18 / len(shown) * HEIGHT / WIDTH / 1.1))
person_maps = (
    (
        F.interpolate(
            torch.from_numpy(masks[shown]).float()[:, None], (MAP_HEIGHT, MAP_WIDTH), mode="area"
        )[:, 0]
        > 0.5
    )
    .cpu()
    .numpy()
)
for column, i in enumerate(shown):
    axes[0, column].imshow(frames[i])
    axes[0, column].set_title(f"piece {segment[i]}, frame {frame_indices[i]}", fontsize=9)
    axes[1, column].imshow(normal_colours(surface.normals[i].astype(np.float32)))
    for row, depth_map in ((2, surface.point_depth[i]), (3, surface.depth[i])):
        shown_depth = np.where(person_maps[column], depth_map.astype(np.float32), np.nan)
        axes[row, column].imshow(shown_depth, cmap="viridis")
for row, label in enumerate(("frame", "normals", "pointmap depth", "Depth Pro depth")):
    axes[row, 0].set_ylabel(label)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## Part 3: the body, shape and pose in turns

The body fit of the turntable (one pose and small movements) does not fit someone moving
freely, and the motion fit poses a shape it is given. So the two alternate: the motion fit
finds a full pose per frame of each piece for the current shape (keypoints, hands, then the
silhouette away from the ignored pixels; smooth motion, knees that bend backwards), then the
shape fit changes Anny's phenotype and local changes under those poses, over all pieces, the
poses moving a little with it. Anny starts in the middle of every phenotype range;
`BODY_HEIGHT` sets the size.

In [ ]:
def tensor(array):
    return torch.from_numpy(np.asarray(array)).float().to(device)


body = AnnyBody().to(device)
corrected = torch.zeros(body.num_bones, 1, device=device)
corrected[[body.bone(n) for n in POSED_BONES if not n.startswith("finger")] + [0]] = 1
mask_width, mask_height = mask_size((WIDTH, HEIGHT), motion_config.resolution)


def packed(flags):
    # (N, H, W) bool -> (1, N, h, ceil(w / 8)) packed at the motion fit's mask size
    small = F.interpolate(tensor(flags)[:, None], (mask_height, mask_width), mode="area")[:, 0]
    return np.packbits((small > 0.5).cpu().numpy(), axis=-1)[None]


# The confident keypoints (Sapiens2's) lifted onto the pointmap's depth: the motion fit starts
# from them and keeps their order in depth.
keypoint_depths = lift_keypoints(
    keypoints, surface.point_depth, (WIDTH, HEIGHT), MIN_KEYPOINT_CONFIDENCE
)
print(
    f"keypoints lifted onto the pointmap: {np.isfinite(keypoint_depths).sum(-1).mean():.1f} "
    f"of 17 per frame; {lifted_frames(keypoint_depths).sum()} of {FRAMES} frames start from them"
)


# All pieces in one motion fit, each as a "person" of its own (independent: they neither touch
# nor hide each other): one step for all of them instead of a fit each. Shorter pieces are padded
# with copies of their last frame, which agree with it (anything else would pull on it through
# the smoothness terms); the copies are dropped from the results.
piece_rows = [np.flatnonzero(segment == piece) for piece in range(SEGMENTS)]
LONGEST = max(len(rows) for rows in piece_rows)


def at_mask_size(maps):
    # (N, h, w, C) float16 maps at the motion fit's mask size, bilinear
    small = F.interpolate(
        torch.from_numpy(maps).float().permute(0, 3, 1, 2),
        (mask_height, mask_width),
        mode="bilinear",
    )
    return small.permute(0, 2, 3, 1).numpy()


# The estimated normals (renormalised) and the two depths, at the mask size.
mask_normals = at_mask_size(surface.normals)
length = np.linalg.norm(mask_normals, axis=-1, keepdims=True)
mask_normals = np.where(length > 0.5, mask_normals / np.maximum(length, 1e-3), 0).astype(np.float16)
mask_depths = at_mask_size(np.stack([surface.point_depth, surface.depth], axis=-1))
mask_depths = np.moveaxis(mask_depths, -1, 1).astype(np.float16)  # (N, D, h, w)
del length


def by_piece(values):
    # (SEGMENTS, LONGEST, ...) each piece's rows of (N, ...) values, padded with its last row
    out = np.empty((SEGMENTS, LONGEST, *values.shape[1:]), dtype=values.dtype)
    for piece, rows in enumerate(piece_rows):
        out[piece] = values[np.concatenate([rows, np.repeat(rows[-1:], LONGEST - len(rows))])]
    return out


def packed_by_piece(flags):
    # packed masks per piece, padded with its last frame's
    out = []
    for rows in piece_rows:
        part = packed(flags[rows])[0]
        out.append(np.concatenate([part, np.repeat(part[-1:], LONGEST - len(part), axis=0)]))
    return np.stack(out)


observations = Observations(
    frame_indices=np.arange(LONGEST),
    fps=fps,
    image_size=np.array([WIDTH, HEIGHT]),
    mask_size=np.array([mask_width, mask_height]),
    masks=packed_by_piece(masks),
    boxes=by_piece(boxes),
    keypoints=by_piece(keypoints),
    hands=by_piece(hands),
    blendshapes=np.zeros((SEGMENTS, LONGEST, 0), np.float32),
    blendshape_names=np.array([], str),
    ignored=packed_by_piece(ignored),
    keypoint_depths=by_piece(keypoint_depths),
    extra_keypoints=by_piece(extra),
    normals=by_piece(mask_normals),
    depths=by_piece(mask_depths),
)


def fit_poses(shape):
    """All pieces' motion for one shape, in the frames' order: rotations, translations, IoU,
    keypoint error."""
    fit = fit_motion(
        observations, body, shape, [], motion_config, fitting_config.silhouette_inset,
        independent=True,
    )  # fmt: skip
    out = []
    for key in ("rotations", "translation", "silhouette_iou", "keypoint_error"):
        values = getattr(fit, key)
        joined = np.empty((FRAMES, *values.shape[2:]), values.dtype)
        for piece, rows in enumerate(piece_rows):
            joined[rows] = values[piece, : len(rows)]
        out.append(joined)
    return tuple(out)


def body_fit_of(phenotype, local_changes, rotations, translation, iou, error):
    return BodyFit(
        frame_indices=frame_indices,
        phenotype=np.asarray(phenotype, np.float32),
        local_changes=np.asarray(local_changes, np.float32),
        rotations=np.asarray(rotations, np.float32),
        translation=np.asarray(translation, np.float32),
        fov_y_degrees=FOV_Y_DEGREES,
        image_size=np.array([WIDTH, HEIGHT]),
        silhouette_iou=np.asarray(iou, np.float32),
        keypoint_error=np.asarray(error, np.float32),
    )


def body_zeros():
    # a zero pose and translation for one body
    zeros = torch.zeros(1, body.num_bones, 3, device=device)
    return zeros, zeros[:, 0]


def fit_shape(shape, poses, learn_shape=True, correction_weight=CORRECTION_WEIGHT, used=None):
    """Anny's phenotype and local changes for the motion fit's poses, which move a little too;
    with learn_shape False only the poses move. Against the keypoints, hands and outlines, and
    the estimated normals and depth, held to her MEASUREMENTS; only on the (N,) ``used`` frames
    (all where None), the others' poses kept as they are."""
    used = np.ones(FRAMES, bool) if used is None else used
    fitting = fitting_config
    scale = fitting.resolution / max(WIDTH, HEIGHT)
    camera = Camera.fixed(WIDTH, HEIGHT, FOV_Y_DEGREES).scaled(scale).to(device)
    size = (camera.height, camera.width)
    target_masks = F.interpolate(tensor(masks)[:, None], size, mode="area")[:, 0]
    compared = (F.interpolate(tensor(ignored)[:, None], size, mode="area")[:, 0] < 0.5).float()
    compared = compared * tensor(used)[:, None, None]
    # the surface is compared inside the mask, away from its edge and the ignored pixels
    hard = (target_masks > 0.5).float()[:, None]
    band = max(1, EDGE_BAND // 2)
    surface_region = (-F.max_pool2d(-hard, 2 * band + 1, stride=1, padding=band))[:, 0] * compared
    points = tensor(np.nan_to_num(keypoints))
    confidence = points[..., 2] * (points[..., 2] >= MIN_KEYPOINT_CONFIDENCE)
    confidence = confidence * tensor(used)[:, None]
    hand_points = tensor(np.nan_to_num(hands))
    hand_confidence = hand_points[..., 2] * (hand_points[..., 2] >= MIN_HAND_CONFIDENCE)
    hand_confidence = hand_confidence * tensor(used)[:, None, None]
    rotations, translation = tensor(poses[0]), tensor(poses[1])
    used_frames = torch.from_numpy(np.flatnonzero(used)).to(device)

    phenotype = torch.nn.Parameter(torch.logit(tensor(shape.phenotype).clamp(1e-3, 1 - 1e-3)))
    changes = torch.nn.Parameter(torch.atanh(tensor(shape.local_changes).clamp(-0.999, 0.999)))
    phenotype.requires_grad_(learn_shape)
    # the performer's known values stay as the config gives them
    held = torch.tensor(
        [label in KNOWN_PHENOTYPE for label in body.phenotype_labels], device=device
    )
    if learn_shape:
        phenotype.register_hook(lambda grad: grad * ~held)
    changes.requires_grad_(learn_shape)
    turn = torch.nn.Parameter(torch.zeros_like(rotations))
    shift = torch.nn.Parameter(torch.zeros_like(translation))
    bone_indices, bone_weights = sparsify_weights(body.skin_weights, BONE_INFLUENCES)

    def posed():
        out = body(
            torch.sigmoid(phenotype)[None],
            torch.tanh(changes)[None],
            rotations + turn * corrected,
            translation + shift,
        )
        vertices = linear_blend_skinning(
            out["rest_vertices"], bone_indices, bone_weights, out["skinning"]
        )
        return out, vertices

    with torch.no_grad():
        out, _ = posed()
        rest = out["rest_vertices"][0]
        samples = SurfaceSamples.draw(rest, body.faces, motion_config.surface_samples)
        hand_joints = HandJoints(body, rest, out["rest_bone_poses"][0])
        # how far behind the skin over each keypoint it lies, as in the motion fit
        insets = torch.cdist(body.keypoint_regressor.regression_weights @ rest, rest).amin(-1)
        limits = JointLimits(body, out["rest_bone_poses"][0])
        proxies = BodyProxies.build(body, rest)
        far_pairs = proxies.far_pairs(body)
        extra_vertices = surface_keypoint_vertices(body, rest, out["rest_bone_poses"][0])
        importance = torch.ones(17, device=device)
        importance[:5] = FACE_SHARE  # the face counts as one point
    extra_points = tensor(np.nan_to_num(extra))
    extra_confidence = extra_points[..., 2] * (extra_points[..., 2] >= MIN_KEYPOINT_CONFIDENCE)
    extra_confidence = extra_confidence * tensor(used)[:, None]
    lifted = tensor(np.where(used[:, None], keypoint_depths, np.nan))
    inset = fitting.silhouette_inset * samples.radius

    def terms():
        out, vertices = posed()
        heads = out["rest_bone_poses"][0, :, :, 3]
        joints = (out["skinning"] @ heads.unsqueeze(-1))[..., :3, 0]
        screen = camera.screen(body.keypoint_regressor({"vertices": vertices}))
        projected = screen[..., :2]
        silhouettes = render_silhouette(
            samples.points(vertices, body.faces, inset), samples.radius, camera
        )
        z = out["rest_vertices"][0, :, 2]
        pick = used_frames[torch.randperm(len(used_frames), device=device)[:SURFACE_FRAMES]]
        surface_ = surface_terms(
            render_surface(raster_context, vertices[pick], body.faces, camera, NEAR, FAR),
            *surface_targets(pick, size),
            surface_region[pick],
        )
        return {
            "normals": SURFACE_WEIGHTS["normals"] * surface_["normals"],
            "depth": SURFACE_WEIGHTS["depth"] * surface_["depth"],
            "keypoint_depth": SURFACE_WEIGHTS["keypoint_depth"]
            * keypoint_depth_loss(screen[..., 2], lifted, insets, importance),
            "height": fitting.height_weight * ((z.max() - z.min()) / BODY_HEIGHT - 1).square(),
            "measurements": MEASUREMENT_WEIGHT
            * measurement_loss(measure(out["rest_vertices"][0], outlines[0]), MEASUREMENTS)
            if MEASUREMENTS is not None and learn_shape
            else z.new_zeros(()),
            "keypoints": fitting.keypoint_weight
            * keypoint_loss(projected, points[..., :2] * scale, confidence, camera.height),
            "extra_keypoints": fitting.keypoint_weight
            * keypoint_loss(
                camera.project(vertices[:, extra_vertices]),
                extra_points[..., :2] * scale,
                extra_confidence,
                camera.height,
            ),
            "hands": fitting.hand_weight
            * keypoint_loss(
                camera.project(hand_joints(vertices, joints)),
                hand_points[..., :2] * scale,
                hand_confidence,
                camera.height,
            ),
            "silhouette": fitting.silhouette_weight
            * silhouette_loss(silhouettes, target_masks, compared),
            "self_contact": motion_config.self_contact_weight
            * self_overlap(proxies.place(out["skinning"]), far_pairs, motion_config.contact_slack),
            "joint_limits": motion_config.joint_limit_weight
            * limits(rotations + turn * corrected).mean(),
            "pose_prior": correction_weight
            * (turn.square().sum((1, 2)).mean() + (10 * shift).square().sum(-1).mean()),
            "smoothness": fitting.smoothness_weight
            * (
                turn.diff(dim=0)[same_segment].square().sum((1, 2)).mean()
                + (10 * shift).diff(dim=0)[same_segment].square().sum(-1).mean()
            ),
            "shape_prior": fitting.local_change_weight * torch.tanh(changes).square().sum(),
        }

    groups_ = [{"params": [turn, shift], "lr": SHAPE_LEARNING_RATES["pose"]}]
    if learn_shape:
        groups_.append({"params": [phenotype, changes], "lr": SHAPE_LEARNING_RATES["shape"]})
    optimizer = torch.optim.Adam(groups_)
    measure, outlines = BodyMeasurements(body), [None]
    for iteration in range(SHAPE_ITERATIONS):
        if MEASUREMENTS is not None and learn_shape and iteration % 25 == 0:
            with torch.no_grad():  # where the bust, waist and hips are, found again now and then
                outlines[0] = measure.levels(posed()[0]["rest_vertices"][0])
        values = terms()
        optimizer.zero_grad()
        sum(values.values()).backward()
        optimizer.step()
        if iteration % 100 == 0 or iteration == SHAPE_ITERATIONS - 1:
            print(
                f"{'shape' if learn_shape else 'pose'} {iteration:4d}: "
                + ", ".join(f"{k} {v.item():.2e}" for k, v in values.items())
            )
    with torch.no_grad():
        # its own scores: IoU outside the ignored pixels, mean keypoint error in pixels
        _, vertices = posed()
        silhouettes = render_silhouette(
            samples.points(vertices, body.faces, inset), samples.radius, camera
        )
        seen = compared > 0.5
        drawn, mask_ = (silhouettes > 0.5) & seen, (target_masks > 0.5) & seen
        iou = (drawn & mask_).sum((1, 2)) / (drawn | mask_).sum((1, 2)).clamp_min(1)
        projected = camera.project(body.keypoint_regressor({"vertices": vertices})) / scale
        sure = confidence > 0
        distance = (projected - points[..., :2]).norm(dim=-1) * sure
        error = torch.where(sure.any(-1), distance.sum(-1) / sure.sum(-1).clamp_min(1), torch.nan)
        return body_fit_of(
            torch.sigmoid(phenotype).cpu().numpy(),
            torch.tanh(changes).cpu().numpy(),
            (rotations + turn * corrected).cpu().numpy(),
            (translation + shift).cpu().numpy(),
            iou.cpu().numpy(),
            error.cpu().numpy(),
        )


body_path = out_dir / "body.npz"
if "body" in REUSE and body_path.exists():
    body_fit = BodyFit.load(body_path)
    print("body fit read from", body_path)
else:
    started = time.time()
    zero_pose = np.zeros((FRAMES, body.num_bones, 3), np.float32)
    zeros3 = np.zeros((FRAMES, 3), np.float32)
    nans = np.full(FRAMES, np.nan, np.float32)
    start_phenotype = body.default_phenotype()[0].cpu().numpy()
    for label, value in KNOWN_PHENOTYPE.items():
        start_phenotype[body.phenotype_labels.index(label)] = value
    start_changes = np.zeros(len(body.local_change_labels), np.float32)
    if MEASUREMENTS is not None:  # shaped to her measurements
        shaped, changes_ = fit_measurements(
            body, tensor(start_phenotype), MEASUREMENTS, held=tuple(KNOWN_PHENOTYPE)
        )
        start_phenotype, start_changes = shaped.cpu().numpy(), changes_.cpu().numpy()
        with torch.no_grad():
            measured_ = BodyMeasurements(body)(
                body(shaped[None], changes_[None], *body_zeros())["rest_vertices"][0]
            )
        print(
            "shaped to her measurements: "
            + ", ".join(
                f"{k} {measured_[k].item():.3f} (hers {v})" for k, v in MEASUREMENTS.items()
            )
        )
    print(
        "shape starts from Anny's average and KNOWN_PHENOTYPE:",
        dict(zip(body.phenotype_labels, start_phenotype.round(2), strict=True)),
    )
    shape = body_fit_of(
        start_phenotype,
        start_changes,
        zero_pose,
        zeros3,
        nans,
        nans,
    )
    for round_ in range(SHAPE_ROUNDS):
        poses = fit_poses(shape)
        print(
            f"round {round_ + 1}: motion fit, IoU {np.nanmean(poses[2]):.3f}, "
            f"keypoints {np.nanmean(poses[3]):.1f} px"
        )
        if round_ == 0:  # the frames her shape is fitted to
            dropped = badly_fitted(poses[2], DROP_SPREAD, DROP_MOST)
            worst = np.where(dropped, poses[2], -np.inf).max() if dropped.any() else np.nan
            print(
                f"{dropped.sum()} of {FRAMES} frames fitted much worse than the rest (IoU "
                f"{worst:.2f} or less) left out of the shape fit"
            )
        shape = fit_shape(shape, poses, used=~dropped)
    poses = fit_poses(shape)
    print(
        f"last motion fit: IoU {np.nanmean(poses[2]):.3f}, keypoints {np.nanmean(poses[3]):.1f} px"
    )
    # The poses pulled onto the normals and depth too, the shape kept.
    body_fit = fit_shape(shape, poses, learn_shape=False, correction_weight=POSE_CORRECTION_WEIGHT)
    body_fit.save(body_path)
    timed("body fit", started)
# Left out of the refinement too: decided on the body fit's own IoU (also where it is reused).
dropped = badly_fitted(body_fit.silhouette_iou, DROP_SPREAD, DROP_MOST)
print(
    f"body: IoU {np.nanmean(body_fit.silhouette_iou):.3f} (outside the ignored pixels), "
    f"keypoints {np.nanmean(body_fit.keypoint_error):.1f} px; on the {(~dropped).sum()} frames "
    f"kept: IoU {np.nanmean(body_fit.silhouette_iou[~dropped]):.3f}, {dropped.sum()} left out"
)

The fit per frame, and its silhouette over the mask on a few frames: white where both are,
red where only the mask is, cyan where only the body is. The ignored pixels are grey. Then the
fitted body's normals under the estimated ones.

In [ ]:
with torch.no_grad():
    phenotype = tensor(body_fit.phenotype)[None]
    changes = tensor(body_fit.local_changes)[None]
    out = body(
        phenotype, changes, tensor(body_fit.rotations[picks]), tensor(body_fit.translation[picks])
    )
    bone_indices, bone_weights = sparsify_weights(body.skin_weights, BONE_INFLUENCES)
    vertices = linear_blend_skinning(
        out["rest_vertices"], bone_indices, bone_weights, out["skinning"]
    )
    camera = Camera.fixed(WIDTH, HEIGHT, FOV_Y_DEGREES).to(device)
    drawn = [
        rasterize(camera.screen(v), body.faces, WIDTH, HEIGHT).mask().cpu().numpy()
        for v in vertices
    ]

    map_camera = Camera.fixed(WIDTH, HEIGHT, FOV_Y_DEGREES).scaled(MAP_WIDTH / WIDTH).to(device)
    fitted_normals = render_surface(raster_context, vertices, body.faces, map_camera, NEAR, FAR)
    fitted_normals = (fitted_normals.normals * fitted_normals.covered[..., None]).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 3))
for piece in range(SEGMENTS):
    picked = np.flatnonzero(segment == piece)
    axes[0].plot(picked, body_fit.silhouette_iou[picked])
    axes[1].plot(picked, body_fit.keypoint_error[picked])
axes[0].set_title("silhouette IoU per frame (colours: pieces)")
axes[1].set_title("keypoint error per frame (px)")
plt.show()
fig, axes = plt.subplots(1, len(picks), figsize=(18, 18 / len(picks) * HEIGHT / WIDTH))
for ax, i, rendered in zip(axes, picks, drawn, strict=True):
    picture = overlay(masks[i], rendered)
    picture[ignored[i]] = 110
    ax.imshow(picture)
    ax.set_title(
        f"piece {segment[i]}, frame {frame_indices[i]}: IoU {body_fit.silhouette_iou[i]:.3f}",
        fontsize=9,
    )
    ax.axis("off")
plt.show()
fig, axes = plt.subplots(2, len(picks), figsize=(18, 2 * 18 / len(picks) * HEIGHT / WIDTH))
for column, i in enumerate(picks):
    axes[0, column].imshow(normal_colours(surface.normals[i].astype(np.float32)))
    axes[0, column].set_title(f"estimated normals, frame {frame_indices[i]}", fontsize=9)
    axes[1, column].imshow(normal_colours(fitted_normals[column]))
    axes[1, column].set_title("the fitted body's", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.show()

## Part 4: the head, ICT-FaceKit fitted to the face landmarks

The fitting stage's head fit, on this video's face, each piece with `MIN_FACE_FRAMES` faces as a
video of its own sharing the identity: 68 landmarks, the irises, and the outline above the chin. Its outline is the face as Sapiens2 labels it (`FACE_PARTS`), and the rest of
the person (hair, a hand before the face) is ignored, so neither pulls the head. Without a
turntable the head is seen from fewer sides; the identity prior keeps the rest plausible.

In [ ]:
model = IctFaceKit(config.body_models.ict_facekit).to(device)
E_HEAD = model.num_expressions
heads_dir = out_dir / "heads"
if "head" in REUSE and heads_dir.exists():
    head_fits = {int(path.stem): HeadFit.load(path) for path in sorted(heads_dir.glob("*.npz"))}
    print("head fits read from", heads_dir)
else:
    started = time.time()
    videos = {}
    for piece in range(SEGMENTS):
        picked = np.flatnonzero(segment == piece)
        if found_faces[picked].sum() < MIN_FACE_FRAMES:
            continue
        videos[piece] = (
            Preprocessed(
                frame_indices=frame_indices[picked],
                masks=face[picked],
                boxes=boxes[picked],
                face_landmarks=face_landmarks[picked],
                ignored=(masks[picked] & ~face[picked]) | ignored[picked],
            ),
            (WIDTH, HEIGHT),
        )
    if not videos:
        raise LookupError("no piece shows a face often enough: nothing to fit the head to")
    head_fits = fit_heads(videos, model, fitting_config, outlier_factor=FACE_OUTLIER)
    heads_dir.mkdir(exist_ok=True)
    for old in heads_dir.glob("*.npz"):
        old.unlink()
    for piece, fit in head_fits.items():
        fit.save(heads_dir / f"{piece}.npz")
    timed("head fit", started)

# The pieces' fits joined over all frames; pieces without one keep a neutral face.
head_identity = next(iter(head_fits.values())).identity
head_expressions = np.zeros((FRAMES, E_HEAD), np.float32)
head_landmark_error = np.full(FRAMES, np.nan)
head_iou = np.full(FRAMES, np.nan)
for piece, fit in head_fits.items():
    picked = segment == piece
    head_expressions[picked] = fit.expressions
    head_landmark_error[picked] = fit.landmark_error
    head_iou[picked] = fit.silhouette_iou
print(
    f"head: {len(head_fits)} of {SEGMENTS} pieces; landmarks {np.nanmean(head_landmark_error):.1f}"
    f" px, IoU above the chin {np.nanmean(head_iou):.3f}"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 3))
axes[0].plot(head_landmark_error)
axes[0].set_title("landmark error per frame (px; gaps: no face)")
axes[1].plot(head_expressions.max(axis=1))
axes[1].set_title("largest expression weight per frame")
plt.show()

## Part 5: the avatar, the head on the body

As the fitting stage builds it: the head model matched to Anny's head, its placement refined
against the video (outline above the shoulders, away from the ignored pixels, and the nose
and eyes), the two joined at the neck, the genitals generated.

In [ ]:
avatar_path, placement_path = out_dir / "avatar.npz", out_dir / "placement.npz"
assert config.body_models.subdivisions == 0, "the refinement below needs the unsubdivided mesh"
body_rest = rest_body(body, body_fit.phenotype, body_fit.local_changes)
head_rest = rest_head(model, head_identity)
if "avatar" in REUSE and avatar_path.exists():
    fitted_avatar = Avatar.load(avatar_path)
    stored = np.load(placement_path)
    placement = HeadPlacement(
        float(stored["scale"]), tensor(stored["rotation"]), tensor(stored["translation"])
    )
    print("avatar read from", avatar_path)
else:
    started = time.time()
    placement = match_heads(body, body_rest, model, head_rest)
    body_input = Preprocessed(
        frame_indices=frame_indices,
        masks=masks,
        boxes=boxes,
        keypoints=keypoints,
        hands=hands,
        face_landmarks=face_landmarks,
        ignored=ignored,
    )
    placement = place_head(
        placement,
        model,
        head_rest,
        body,
        body_rest["vertices"],
        body_fit,
        body_input,
        fitting_config,
    )
    fitted_avatar = build_avatar(body, body_rest, model, head_rest, placement, config.body_models)
    fitted_avatar.save(avatar_path)
    np.savez(
        placement_path,
        scale=placement.scale,
        rotation=placement.rotation.cpu().numpy(),
        translation=placement.translation.cpu().numpy(),
    )
    timed("avatar", started)
print(
    f"head scaled by {placement.scale:.3f}; {len(fitted_avatar.rest_vertices)} vertices, "
    f"{len(fitted_avatar.faces)} triangles"
)

## Part 6: refinement, the frames

From here on notebook 09's analysis-by-synthesis, on this video's frames at `RESOLUTION`. The
colour is compared inside the mask eroded by `EDGE_BAND`, away from the ignored pixels and
without the hands; the outline away from the ignored pixels.

In [ ]:
scale = min(RESOLUTION, max(WIDTH, HEIGHT)) / max(WIDTH, HEIGHT)
camera = Camera.fixed(WIDTH, HEIGHT, FOV_Y_DEGREES).scaled(scale).to(device)
size = (camera.height, camera.width)
images = torch.from_numpy(frames).to(device).permute(0, 3, 1, 2).float() / 255
images = F.interpolate(images, size, mode="area").permute(0, 2, 3, 1).contiguous()
mask_targets = F.interpolate(tensor(masks)[:, None], size, mode="area")[:, 0]
compared = (F.interpolate(tensor(ignored)[:, None], size, mode="area")[:, 0] < 0.5).float()
hard = (mask_targets > 0.5).float()[:, None]
inner = (-F.max_pool2d(-hard, 2 * EDGE_BAND + 1, stride=1, padding=EDGE_BAND))[:, 0] * compared

keypoint_targets = tensor(np.nan_to_num(keypoints))
keypoint_confidence = keypoint_targets[..., 2] * (
    keypoint_targets[..., 2] >= MIN_KEYPOINT_CONFIDENCE
)
keypoint_targets = keypoint_targets[..., :2] * scale
face_targets = landmark_targets(face_landmarks, scale, model, device)
# the faces the head fit left out are left out here too
left_out = found_faces & np.isin(segment, list(head_fits)) & np.isnan(head_landmark_error)
face_targets.weights[torch.from_numpy(left_out).to(device)] = 0
print(f"{left_out.sum()} faces left out of the landmarks")

# The frames fitted badly are neither trained on nor baked from, only scored (as held out).
left_out = torch.from_numpy(dropped)
held_out = torch.zeros(FRAMES, dtype=torch.bool)
held_out[HOLD_OUT // 2 :: HOLD_OUT] = True
held_out &= ~left_out
chunks = [(c, False) for c in torch.nonzero(~held_out & ~left_out)[:, 0].split(CHUNK)]
chunks += [(c, True) for c in torch.nonzero(held_out | left_out)[:, 0].split(CHUNK)]
trainable = torch.nonzero(~left_out)[:, 0]
print(f"{FRAMES} frames at {camera.width}x{camera.height}, {int(held_out.sum())} held out")

## The model

Notebook 09's: per frame the face's blendshapes, then linear blend skinning by the body fit's
bone rotations plus that frame's corrections (not the fingers'); albedo × the light the
normal gets. Two things are new, both on the head: its **identity** and smooth **shape
changes** (notebook 08's broad layer) are learned too. Both are changes of the head model,
placed on the body as the head is, and fade out towards the neck seam, so the join stays
closed. The 17 body keypoints come straight from the bones (as in the motion fit), the head's
68 landmarks from its vertices.

In [ ]:
avatar = fitted_avatar
faces = torch.from_numpy(avatar.faces).long().to(device)
tri = faces.int().contiguous()
face_textures = torch.from_numpy(avatar.face_textures).long().to(device)
face_parts = torch.from_numpy(avatar.face_parts).long().to(device)
corner_tri = torch.arange(3 * len(tri), device=device, dtype=torch.int32).reshape(-1, 3)
corner_uvs = torch.from_numpy(avatar.uvs).float().to(device).reshape(-1, 2).clone()
corner_uvs[:, 1] = 1 - corner_uvs[:, 1]
corner_uvs = corner_uvs[None].contiguous()
bone_indices = torch.from_numpy(avatar.bone_indices).long().to(device)
bone_weights = torch.from_numpy(avatar.bone_weights).float().to(device)
bind_poses = torch.from_numpy(avatar.bind_poses).float().to(device)
context = dr.RasterizeCudaContext()

HEAD_START = int(avatar.head_start)
HEAD_COUNT = len(model.neutral)
HEAD = slice(HEAD_START, HEAD_START + HEAD_COUNT)
rest = tensor(avatar.rest_vertices)
basis = tensor(avatar.expression_basis)
landmark_indices = HEAD_START + model.landmarks

# The body keypoints, from the skinning matrices.
anny_bones, anny_weights = sparsify_weights(body.skin_weights, BONE_INFLUENCES)
keypoint_weights = skinned_point_weights(
    body.keypoint_regressor.regression_weights,
    body_rest["vertices"],
    anny_bones,
    anny_weights,
    body.num_bones,
)


def averaging_matrix(mesh_faces, count):
    # (count, count) sparse: each vertex's mean with its neighbours
    edges = torch.cat([mesh_faces[:, [0, 1]], mesh_faces[:, [1, 2]], mesh_faces[:, [2, 0]]])
    loops = torch.arange(count, device=device)[:, None].repeat(1, 2)
    edges = torch.unique(torch.cat([edges, edges.flip(1), loops]), dim=0)
    degree = torch.bincount(edges[:, 0], minlength=count).float()
    return torch.sparse_coo_tensor(
        edges.T, 1 / degree[edges[:, 0]], (count, count), check_invariants=True
    ).coalesce()


def smooth(values, matrix, steps):
    for _ in range(steps):
        values = torch.sparse.mm(matrix, values)
    return values


MILLIMETRE = 1e-3

# The body's shape changes (notebook 09).
body_averaging = averaging_matrix(faces[(faces < HEAD_START).all(dim=1)], HEAD_START)
with torch.no_grad():
    free = torch.ones(HEAD_START, 1, device=device)
    hand_vertices = torch.unique(faces[face_parts == HANDS])
    free[hand_vertices[hand_vertices < HEAD_START]] = 0
    nearest = torch.cat(
        [
            torch.cdist(part, rest[HEAD_START:]).min(dim=1).values
            for part in rest[:HEAD_START].split(4096)
        ]
    )
    free[nearest < FROZEN_NEAR] = 0
offset_values = torch.nn.Parameter(torch.zeros(HEAD_START, 3, device=device))


def body_offsets():
    # (HEAD_START, 3) millimetres
    return smooth(offset_values * free, body_averaging, SMOOTHING_STEPS) * free


# The head's identity and shape changes (notebook 08), in the head model's coordinates.
SKIN_COUNT = ict.SKIN.stop
skin_faces = model.faces[model.faces.max(dim=1).values < SKIN_COUNT]
head_averaging = averaging_matrix(skin_faces, SKIN_COUNT)
SOCKETS_END = 14062  # mouth socket and eye sockets: joined to the skin at the lips and lids
INTERIOR = slice(SOCKETS_END, ict.EYEBALLS.start)  # gums, tongue, teeth: they keep their shape
with torch.no_grad():
    nearest_skin = torch.cat(
        [
            torch.cdist(part, model.neutral[:SKIN_COUNT]).argmin(dim=1)
            for part in model.neutral[SKIN_COUNT:].split(4096)
        ]
    )


def spread(skin):
    # (SKIN_COUNT, 3) changes of the skin -> (HEAD_COUNT, 3) changes of every head vertex
    parts = [
        skin,
        skin[nearest_skin[: SOCKETS_END - SKIN_COUNT]],
        skin.new_zeros(INTERIOR.stop - INTERIOR.start, 3),
    ]
    for eyeball in ict.EYEBALL_ORDER:
        followed = nearest_skin[eyeball.start - SKIN_COUNT : eyeball.stop - SKIN_COUNT]
        parts.append(
            skin[followed].mean(dim=0, keepdim=True).expand(eyeball.stop - eyeball.start, 3)
        )
    return torch.cat(parts)


no_expressions = torch.zeros(1, E_HEAD, device=device)
identity = torch.nn.Parameter(tensor(head_identity))
with torch.no_grad():
    fitted_head = model(identity[None], no_expressions)[0]
    head_z = rest[HEAD, 2]
    seam = head_z[model.landmarks[ICT_CHIN]] - config.body_models.neck_seam
    fade = ((head_z - seam) / config.body_models.neck_seam).clamp(0, 1)[:, None]
head_offset_values = torch.nn.Parameter(torch.zeros(SKIN_COUNT, 3, device=device))
head_linear = (placement.scale * placement.rotation).float().to(device)


def head_offsets():
    # (SKIN_COUNT, 3) millimetres, in the head model's axes
    return smooth(head_offset_values, head_averaging, HEAD_SMOOTHING_STEPS)


def head_change():
    # (HEAD_COUNT, 3) metres on the body: the identity's and the shape changes' difference
    local = model(identity[None], no_expressions)[0] - fitted_head
    local = local + spread(head_offsets()) * MILLIMETRE
    return fade * (local @ head_linear.T)


def rest_vertices():
    return torch.cat(
        [
            rest[:HEAD_START] + body_offsets() * MILLIMETRE,
            rest[HEAD] + head_change(),
            rest[HEAD.stop :],
        ]
    )


# Per frame: corrections of the bones' rotations and the root's position, and the face.
fit_rotations = tensor(body_fit.rotations)
fit_translations = tensor(body_fit.translation)
rotation_corrections = torch.nn.Parameter(torch.zeros_like(fit_rotations))
translation_corrections = torch.nn.Parameter(torch.zeros_like(fit_translations))
start_weights = np.clip(head_expressions, 1e-4, 1 - 1e-4)
expression_logits = torch.nn.Parameter(tensor(np.log(start_weights / (1 - start_weights))))


def expressions():
    return torch.sigmoid(expression_logits)


joint_limits = JointLimits(body, bind_poses)  # the corrected poses stay within them


def frame_pose(frames):
    rotations = fit_rotations[frames] + rotation_corrections[frames] * corrected
    return rotations, fit_translations[frames] + translation_corrections[frames]


def posed(rest_now, weights, rotations, translations):
    # (B, V, 3) vertices and (B, J, 4, 4) skinning
    skinning = body.skinning(bind_poses, rotations, translations)
    rest_b = rest_now.expand(len(rotations), -1, -1)
    face_change = torch.einsum("be,evc->bvc", weights, basis[:E_HEAD, :HEAD_COUNT])
    rest_b = torch.cat(
        [rest_b[:, :HEAD_START], rest_b[:, HEAD] + face_change, rest_b[:, HEAD.stop :]], 1
    )
    return linear_blend_skinning(rest_b, bone_indices, bone_weights, skinning), skinning


def vertex_normals(vertices):
    corners = vertices[:, faces]
    face_normals = torch.linalg.cross(
        corners[:, :, 1] - corners[:, :, 0], corners[:, :, 2] - corners[:, :, 0]
    )
    normals = torch.zeros_like(vertices)
    for k in range(3):
        normals = normals.index_add(1, faces[:, k], face_normals)
    return F.normalize(normals, dim=-1)


def sh_basis(n):
    x, y, z = n.unbind(-1)
    return torch.stack(
        [
            0.282095 * torch.ones_like(x),
            0.488603 * y,
            0.488603 * z,
            0.488603 * x,
            1.092548 * x * y,
            1.092548 * y * z,
            0.315392 * (3 * z * z - 1),
            1.092548 * x * z,
            0.546274 * (x * x - y * y),
        ],
        dim=-1,
    )


def uniform_light():
    light = torch.zeros(9, 3, device=device)
    light[0] = 1 / 0.282095
    return light


def to_clip(vertices, camera):
    view = vertices @ camera.view[:3, :3].T + camera.view[:3, 3]
    x, y, z = view.unbind(-1)
    K, W, H = camera.K, camera.width, camera.height
    return torch.stack(
        [
            2 * K[0, 0] / W * x + (2 * K[0, 2] / W - 1) * z,
            2 * K[1, 1] / H * y + (2 * K[1, 2] / H - 1) * z,
            (FAR + NEAR) / (FAR - NEAR) * z - 2 * FAR * NEAR / (FAR - NEAR),
            z,
        ],
        dim=-1,
    ).contiguous()


BODY_SHAPE = (BODY_TEXTURE_SIZE, BODY_TEXTURE_SIZE)
HEAD_TILES = ATLAS_TILES
HEAD_SHAPE = (HEAD_TILE_SIZE, HEAD_TILES * HEAD_TILE_SIZE)
with torch.no_grad():
    mean_colour = (images * inner[..., None]).sum((0, 1, 2)) / inner.sum()
    baked = {
        "body": mean_colour.expand(1, *BODY_SHAPE, 3).clone(),
        "head": mean_colour.expand(1, *HEAD_SHAPE, 3).clone(),
    }
light = torch.nn.Parameter(uniform_light())


def render(vertices, camera, body_texture, head_texture, light):
    clip = to_clip(vertices, camera)
    rast, rast_db = dr.rasterize(context, clip, tri, (camera.height, camera.width))
    covered = (rast[..., 3] > 0).float()[..., None]
    face = (rast[..., 3].long() - 1).clamp_min(0)
    uv, uv_da = dr.interpolate(corner_uvs, rast, corner_tri, rast_db=rast_db, diff_attrs="all")
    on_head = (face_textures[face] == HEAD_TEXTURE).float()[..., None]
    mipmapped = {"filter_mode": "linear-mipmap-linear", "max_mip_level": 8}
    from_body = dr.texture(body_texture, uv, uv_da, **mipmapped)
    from_head = dr.texture(head_texture, uv, uv_da, **mipmapped)
    colour = torch.lerp(from_body, from_head, on_head).clamp(0, 1) * covered
    normals, _ = dr.interpolate(vertex_normals(vertices).contiguous(), rast, tri)
    shading = (sh_basis(F.normalize(normals, dim=-1)) @ light).clamp_min(0) * covered
    lit = dr.antialias((colour * shading).contiguous(), rast, clip, tri)
    alpha = dr.antialias(covered.contiguous(), rast, clip, tri)[..., 0]
    part = torch.where(covered[..., 0] > 0, face_parts[face], -1)
    drawn = surface_values(rast, vertices, faces, camera)
    return {
        "surface": drawn,
        "colour": lit.clamp(0, 1),
        "albedo": colour,
        "covered": covered[..., 0],
        "alpha": alpha,
        "part": part,
    }

## Losses and scores

Per frame: colour, outline, body keypoints and face landmarks. Over the video: small, smooth
corrections, a sparse face, small shape changes, the identity prior. PSNR is scored on the
person away from the ignored pixels (hands included), with the outline's IoU there.

In [ ]:
def shared_state(detached):
    state = {
        "rest": rest_vertices(),
        "body_texture": baked["body"],
        "head_texture": baked["head"],
        "light": light,
    }
    return {k: v.detach() for k, v in state.items()} if detached else state


def frame_terms(frames, shared):
    weights = expressions()[frames]
    rotations, translations = frame_pose(frames)
    vertices, skinning = posed(shared["rest"], weights, rotations, translations)
    out = render(vertices, camera, shared["body_texture"], shared["head_texture"], shared["light"])
    region = inner[frames] * out["covered"] * (out["part"] != HANDS).float()
    error = (out["colour"] - images[frames]).abs().mean(-1)
    terms = {
        "photo": (error * region).sum() / region.sum().clamp_min(1),
        "silhouette": silhouette_loss(out["alpha"], mask_targets[frames], compared[frames]),
        "keypoints": keypoint_loss(
            camera.project(skinned_points(skinning, keypoint_weights)),
            keypoint_targets[frames],
            keypoint_confidence[frames],
            camera.height,
        ),
        "landmarks": keypoint_loss(
            camera.project(vertices[:, landmark_indices]),
            face_targets.landmarks[frames],
            face_targets.weights[frames],
            camera.height,
        ),
    }
    terms.update(
        surface_terms(
            out["surface"],
            *surface_targets(frames, size),
            inner[frames] * (out["part"] != HANDS).float(),
        )
    )
    return terms, out


def video_terms():
    return {
        "pose_prior": rotation_corrections.square().sum((1, 2)).mean()
        + (10 * translation_corrections).square().sum(-1).mean(),
        "temporal": second_difference(rotation_corrections)[steady].square().sum((1, 2)).mean()
        + second_difference(10 * translation_corrections)[steady].square().sum(-1).mean()
        + 0.01 * expressions().diff(dim=0)[same_segment].square().sum(-1).mean(),
        "joint_limits": joint_limits(fit_rotations + rotation_corrections * corrected).mean(),
        "sparsity": expressions().sum(-1).mean(),
        "identity_prior": identity.square().sum(),
        "offsets": body_offsets().square().sum(-1).mean(),
        "head_offsets": head_offsets().square().sum(-1).mean(),
    }


@torch.no_grad()
def scores():
    shared = shared_state(detached=True)
    psnr, iou = torch.zeros(FRAMES), torch.zeros(FRAMES)
    for frames, _ in chunks:
        _, out = frame_terms(frames, shared)
        region = inner[frames] * out["covered"]
        mse = ((out["colour"] - images[frames]).square().mean(-1) * region).sum(
            (1, 2)
        ) / region.sum((1, 2)).clamp_min(1)
        psnr[frames] = (-10 * torch.log10(mse)).cpu()
        seen = compared[frames] > 0.5
        drawn, mask = (out["alpha"] > 0.5) & seen, (mask_targets[frames] > 0.5) & seen
        iou[frames] = ((drawn & mask).sum((1, 2)) / (drawn | mask).sum((1, 2)).clamp_min(1)).cpu()
    return {"psnr": psnr.numpy(), "iou": iou.numpy()}


def summary(result, label):
    trained_ = ~held_out.numpy() & ~left_out.numpy()
    parts = (("trained", trained_), ("held out", held_out.numpy()), ("left out", left_out.numpy()))
    for part, chosen_ in parts:
        if not chosen_.any():
            continue
        psnr, iou = result["psnr"][chosen_].mean(), result["iou"][chosen_].mean()
        print(f"{label:>8} {part:>8}: PSNR {psnr:6.2f} dB, IoU {iou:.3f}")

## Baking the albedo

Notebook 09's bake, for the whole head atlas too (there is no head refinement to take it
from): every training frame's pixels inside the eroded mask, away from the ignored ones,
divided by their light, splatted into the texels they show, weighted by how squarely the
surface faces the camera; the gaps filled from coarser levels. Under hair the scalp's texels
get no pixel, so they take the colour around them.

In [ ]:
def push_pull(sums, weights):
    levels = [(sums.permute(2, 0, 1)[None], weights[None, None])]
    while max(levels[-1][1].shape[-2:]) > 1:
        s, w = levels[-1]
        shape = (max(s.shape[-2] // 2, 1), max(s.shape[-1] // 2, 1))
        levels.append((F.interpolate(s, shape, mode="area"), F.interpolate(w, shape, mode="area")))
    s, w = levels[-1]
    value = s / w.clamp_min(1e-8) if w.max() > 0 else s
    for s, w in reversed(levels[:-1]):
        coarser = F.interpolate(value, s.shape[-2:], mode="bilinear", align_corners=False)
        trust = w / (w + FILL_WEIGHT)
        value = trust * s / w.clamp_min(1e-8) + (1 - trust) * coarser
    return value.permute(0, 2, 3, 1).clamp(0, 1).contiguous()


@torch.no_grad()
def bake(shared):
    shapes = {BODY_TEXTURE: BODY_SHAPE, HEAD_TEXTURE: HEAD_SHAPE}
    sums = {t: torch.zeros(h * w, 3, device=device) for t, (h, w) in shapes.items()}
    totals = {t: torch.zeros(h * w, device=device) for t, (h, w) in shapes.items()}
    for frames, held in chunks:
        if held:
            continue
        rotations, translations = frame_pose(frames)
        vertices, _ = posed(shared["rest"], expressions()[frames], rotations, translations)
        clip = to_clip(vertices, camera)
        rast, _ = dr.rasterize(context, clip, tri, (camera.height, camera.width))
        face = rast[..., 3].long() - 1
        uv, _ = dr.interpolate(corner_uvs, rast, corner_tri)
        normals, _ = dr.interpolate(vertex_normals(vertices).contiguous(), rast, tri)
        normals = F.normalize(normals, dim=-1)
        points, _ = dr.interpolate(vertices.contiguous(), rast, tri)
        facing = (normals * F.normalize(camera.position - points, dim=-1)).sum(-1).clamp_min(0)
        shading = sh_basis(normals) @ shared["light"]
        usable_ = (face >= 0) & (inner[frames] > 0.5) & (shading.min(dim=-1).values > MIN_SHADING)
        albedo = images[frames] / shading.clamp_min(MIN_SHADING)
        for texture_id, (height_, width_) in shapes.items():
            picked = usable_ & (face_textures[face.clamp_min(0)] == texture_id)
            x = uv[..., 0][picked] * width_ - 0.5
            y = uv[..., 1][picked] * height_ - 0.5
            x0, y0 = x.floor(), y.floor()
            fx, fy = x - x0, y - y0
            value, weight = albedo[picked], facing[picked].square()
            for dx, dy, share in (
                (0, 0, (1 - fx) * (1 - fy)),
                (1, 0, fx * (1 - fy)),
                (0, 1, (1 - fx) * fy),
                (1, 1, fx * fy),
            ):
                column = (x0 + dx).long().clamp(0, width_ - 1)
                row = (y0 + dy).long().clamp(0, height_ - 1)
                index = row * width_ + column
                sums[texture_id].index_add_(0, index, value * (weight * share)[:, None])
                totals[texture_id].index_add_(0, index, weight * share)
    return {
        t: push_pull(sums[t].view(h, w, 3), totals[t].view(h, w)) for t, (h, w) in shapes.items()
    }


def rebake():
    result = bake(shared_state(detached=True))
    baked["body"] = result[BODY_TEXTURE]
    baked["head"] = result[HEAD_TEXTURE]


rebake()

## Showing frames: whole, and around the head

In [ ]:
@torch.no_grad()
def head_box(vertices, margin=1.6):
    points = camera.project(vertices[HEAD][: ict.SKIN.stop])
    centre = (points.min(0).values + points.max(0).values) / 2
    half = margin * (points.max(0).values - points.min(0).values).max() / 2
    x0, y0 = (centre - half).clamp_min(0).long().tolist()
    x1, y1 = (centre + half).long().tolist()
    return slice(y0, min(y1, camera.height)), slice(x0, min(x1, camera.width))


@torch.no_grad()
def show(title=""):
    shared = shared_state(detached=True)
    fig, axes = plt.subplots(len(picks), 6, figsize=(18, 18 / 6 * len(picks) * HEIGHT / WIDTH))
    for row, i in enumerate(picks):
        frames_ = torch.tensor([int(i)], device=device)
        _, out = frame_terms(frames_, shared)
        rotations, translations = frame_pose(frames_)
        vertices, _ = posed(shared["rest"], expressions()[frames_], rotations, translations)
        crop = head_box(vertices[0])
        image = images[i].cpu().numpy()
        rendered = out["colour"][0].cpu().numpy()
        albedo = out["albedo"][0].cpu().numpy()
        error = np.abs(rendered - image).mean(-1) * (out["covered"][0] * compared[i]).cpu().numpy()
        held = " (held out)" if held_out[i] else ""
        for ax, picture, label in zip(
            axes[row],
            (image, rendered, albedo, error, image[crop], rendered[crop]),
            ("frame", "render", "albedo", "|error|", "head", "head render"),
            strict=True,
        ):
            ax.imshow(
                picture,
                cmap="magma" if picture.ndim == 2 else None,
                vmin=0,
                vmax=0.3 if picture.ndim == 2 else None,
            )
            ax.set_title(f"{frame_indices[i]}{held}: {label}", fontsize=9)
            ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show(title="the fit as it is: albedo baked under a uniform light")

## Optimisation

Notebook 09's phases, with the head's identity and shape changes joining the body's in the
last.

In [ ]:
parameters_of = {
    "light": [light],
    "pose": [rotation_corrections, translation_corrections],
    "expressions": [expression_logits],
    "identity": [identity],
    "offsets": [offset_values],
    "head_offsets": [head_offset_values],
}
groups = {
    "appearance": ["light"],
    "pose": ["light", "pose", "expressions"],
    "shape": ["light", "pose", "expressions", "identity", "offsets", "head_offsets"],
}
history = []


def set_trainable(trainable):
    for group, parameters in parameters_of.items():
        for p in parameters:
            p.requires_grad_(group in trainable)


def backward(loss):
    if loss.requires_grad:
        loss.backward()


def iteration_step(sample):
    # one iteration's gradients from the (n,) sampled frames; the shared values are built once
    # and every chunk renders from detached copies, whose gradients are pushed back through
    # them at the end. Held-out frames train only their own pose and face.
    shared = shared_state(detached=False)
    leaves = {k: v.detach().requires_grad_(v.requires_grad) for k, v in shared.items()}
    held = held_out[sample]
    step_chunks = [(c, False) for c in sample[~held].split(CHUNK)]
    step_chunks += [(c, True) for c in sample[held].split(CHUNK)]
    logged = {}
    for frames, held_chunk in step_chunks:
        used = {k: v.detach() for k, v in leaves.items()} if held_chunk else leaves
        terms, _ = frame_terms(frames, used)
        share = len(frames) / len(sample)
        backward(sum(WEIGHTS[k] * t for k, t in terms.items()) * share)
        for k, t in terms.items():
            logged[k] = logged.get(k, 0.0) + t.detach() * share
    pairs = [(shared[k], leaves[k].grad) for k in shared if leaves[k].grad is not None]
    if pairs:
        torch.autograd.backward([value for value, _ in pairs], [grad for _, grad in pairs])
    regular = video_terms()
    backward(sum(WEIGHTS[k] * t for k, t in regular.items()))
    logged.update(regular)
    return {k: float(t) for k, t in logged.items()}  # one read from the GPU per term


# Per frame: one tensor of each, so Adam would move frames left out of a step by their
# momentum (a random walk that cost 2-4 dB); they are held where they are instead.
per_frame = [rotation_corrections, translation_corrections, expression_logits]


def step_sampled(optimizer, sample):
    left = torch.ones(FRAMES, dtype=torch.bool)
    left[sample] = False
    left = left.to(device)
    kept = []
    for p in per_frame:
        if p.grad is not None:
            p.grad[left] = 0
            kept.append((p, p.detach()[left].clone()))
    optimizer.step()
    with torch.no_grad():
        for p, values in kept:
            p[left] = values


def run_phase(phase):
    started = time.time()
    set_trainable(groups[phase])
    optimizer = torch.optim.Adam(
        [{"params": parameters_of[g], "lr": LEARNING_RATES[g]} for g in groups[phase]]
    )
    generator = torch.Generator().manual_seed(len(history))
    for iteration in range(ITERATIONS[phase]):
        if iteration % REBAKE_EVERY == 0:
            rebake()
        optimizer.zero_grad()
        drawn = torch.randperm(len(trainable), generator=generator)[:STEP_FRAMES]
        sample = trainable[drawn].sort().values
        logged = iteration_step(sample)
        step_sampled(optimizer, sample)
        history.append({"phase": phase, **logged})
        if iteration % 50 == 0 or iteration == ITERATIONS[phase] - 1:
            print(
                f"{phase} {iteration:4d} ({time.time() - started:5.0f} s): "
                + ", ".join(f"{k} {v:.2e}" for k, v in logged.items())
            )
    timed(f"refinement: {phase}", started)

### Phase 1: the light, with the albedo baked again as it changes

In [ ]:
run_phase("appearance")
before = scores()
summary(before, "before")
show(title="after phase 1: fitted poses, baked albedo, learned light")

### Phase 2: + per-frame pose corrections and the face

In [ ]:
run_phase("pose")
summary(scores(), "phase 2")
show(title="after phase 2")

### Phase 3: + the head's identity and the shape changes of body and head

In [ ]:
run_phase("shape")
rebake()
after = scores()
summary(before, "before")
summary(after, "after")
show(title="after phase 3")

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
for ax, key, label in zip(axes, ("psnr", "iou"), ("PSNR (dB)", "IoU"), strict=True):
    ax.plot(before[key], "--", label="before")
    (line,) = ax.plot(after[key], label="after")
    held = np.flatnonzero(held_out.numpy())
    ax.plot(held, after[key][held], "o", color=line.get_color(), markersize=4)
    ax.set_title(label + " per frame (dots: held out)")
    ax.set_xlabel("frame")
axes[0].legend()
plt.show()

with torch.no_grad():
    for name in ("body", "head"):
        plt.figure(figsize=(16 if name == "head" else 8, 4 if name == "head" else 8))
        plt.imshow(baked[name][0].clamp(0, 1).cpu().numpy())
        plt.title(f"{name} albedo")
        plt.axis("off")
        plt.show()

The avatar in its rest pose, turning, under the learned light and under a uniform one (the
albedo), and how far body and head moved from the fits, in millimetres.

In [ ]:
YAWS = np.linspace(0, 2 * np.pi, 6, endpoint=False)


@torch.no_grad()
def turntable(shared, lit=True, colour_by=None):
    count = len(YAWS)
    rotations = torch.zeros(count, body.num_bones, 3, device=device)
    rotations[:, 0, 2] = torch.from_numpy(YAWS).float().to(device)
    vertices, _ = posed(
        shared["rest"],
        torch.zeros(count, E_HEAD, device=device),
        rotations,
        torch.zeros(count, 3, device=device),
    )
    # stand it in front of the camera, whole
    low, high = vertices.amin(dim=(0, 1)), vertices.amax(dim=(0, 1))
    distance = camera.depth_for_height(1.15 * float(high[2] - low[2]), camera.height)
    vertices = vertices - (low + high) / 2 + torch.tensor([0, distance, 0], device=device)
    if colour_by is not None:
        clip = to_clip(vertices, camera)
        rast, _ = dr.rasterize(context, clip, tri, (camera.height, camera.width))
        value, _ = dr.interpolate(colour_by[None, :, None].contiguous(), rast, tri)
        return (value[..., 0] * (rast[..., 3] > 0)).cpu().numpy()
    light_ = shared["light"] if lit else uniform_light()
    rendered = render(vertices, camera, shared["body_texture"], shared["head_texture"], light_)
    return rendered["colour"].cpu().numpy()


final = shared_state(detached=True)
moved = (final["rest"] - rest).norm(dim=-1) / MILLIMETRE
rows = (
    ("learned light", turntable(final)),
    ("albedo", turntable(final, lit=False)),
    ("shape change (mm)", turntable(final, colour_by=moved)),
)
fig, axes = plt.subplots(3, 6, figsize=(18, 14))
for r, (label, pictures) in enumerate(rows):
    for c in range(6):
        scalar = pictures.ndim == 3
        axes[r, c].imshow(
            pictures[c],
            cmap="viridis" if scalar else None,
            vmin=0,
            vmax=float(moved.max()) if scalar else None,
        )
        axes[r, c].axis("off")
    axes[r, 0].set_title(label, loc="left")
plt.tight_layout()
plt.show()
for part, vertices_ in (("body", slice(0, HEAD_START)), ("head", HEAD)):
    print(
        f"{part} shape change: mean {moved[vertices_].mean():.2f} mm, "
        f"max {moved[vertices_].max():.2f} mm"
    )

## Part 7: export for the viewer

As notebook 09 exports it, to `outputs/export/<name>.avatar`: the mesh with the refined body
and head, the baked albedo, the learned light, the lashes and brows, no Gaussians. The eyes'
and brows' colours are measured in the frames that face the camera; where none does, the
eyes are drawn from the texture and the brows get a dark brown.

In [ ]:
with torch.no_grad():
    refined = dataclasses.replace(avatar, rest_vertices=final["rest"].cpu().numpy())
    textures = TextureRecord(
        body=final["body_texture"][0].clamp(0, 1).cpu().numpy(),
        head=final["head_texture"][0].clamp(0, 1).cpu().numpy(),
        body_coverage=np.ones(BODY_SHAPE, np.float32),
        head_coverage=np.ones(HEAD_SHAPE, np.float32),
    )

ict_landmarks = ict_landmark_vertices(config.body_models.ict_facekit)
try:
    eyes = eye_metadata(refined, measure_eyes(frames, face_landmarks), ict_landmarks)
    stock_eyes = STOCK_EYES_AND_MOUTH
except ValueError as error:
    print(f"eyes from the texture: {error}")
    eyes, stock_eyes = {}, False  # the entry is dropped below
try:
    hair_colours = measure_hair_colours(frames, face_landmarks)
except ValueError as error:
    print(f"brows and lashes dark brown: {error}")
    hair_colours = {"brows": np.array([0.18, 0.13, 0.1]), "lashes": np.array([0.08, 0.06, 0.05])}
hair = grow_face_hair(
    refined,
    textures,
    config.body_models.ict_facekit,
    identity.detach().cpu().numpy(),
    np.random.default_rng(0),
)
no_gaussians = GaussianRecord(
    faces=np.zeros(0, np.int64),
    barycentric=np.zeros((0, 3), np.float32),
    offsets=np.zeros((0, 3), np.float32),
    rotations=np.zeros((0, 4), np.float32),
    log_scales=np.zeros((0, 3), np.float32),
    opacity_logits=np.zeros(0, np.float32),
    sh=np.zeros((0, 1, 3), np.float32),
)
file = avatar_arrays(
    refined,
    no_gaussians,
    textures,
    config.training.occlusion_margin,
    eyes,
    hair,
    face_hair_metadata(hair_colours),
)
file.metadata["format_version"] = 4
if not stock_eyes:
    del file.metadata["eyes"], file.metadata["mouth"]
file.metadata["lighting"] = {"sh": light.detach().cpu().numpy().tolist()}

path = config.outputs / "export" / f"{NAME}.avatar"
write_avatar(path, file)
np.savez_compressed(
    out_dir / "refinement.npz",
    frame_indices=frame_indices,
    segment=segment,
    rotation_corrections=rotation_corrections.detach().cpu().numpy(),
    translation_corrections=translation_corrections.detach().cpu().numpy(),
    expressions=expressions().detach().cpu().numpy(),
    identity=identity.detach().cpu().numpy(),
    rest_vertices=final["rest"].cpu().numpy(),
    light=light.detach().cpu().numpy(),
    before_psnr=before["psnr"],
    after_psnr=after["psnr"],
    before_iou=before["iou"],
    after_iou=after["iou"],
    held_out=held_out.numpy(),
)
print(f"wrote {path} ({path.stat().st_size / 1e6:.0f} MB)")

## Where the time went

In [ ]:
total = sum(timings.values())
for step_, seconds in sorted(timings.items(), key=lambda item: -item[1]):
    print(f"{step_:<28}{seconds:>8.0f} s{seconds / total:>7.0%}")
print(f"{'total':<28}{total:>8.0f} s")